# Efficiency–Performance Trade-offs in Neural Sequence Architectures
### Multitask comparative study: BiGRU vs Transformer on Project CodeNet (Python submissions)

This notebook runs the full study end to end: data indexing → tokenization → leakage-safe split →
non-neural baseline → parameter-matched models → efficiency profiling → multi-seed multitask experiment →
Pareto analysis → sequence-length scaling → auto-generated guidance → reproducibility manifest.

**Contributions implemented**
- **C1** — Multitask × architecture interaction (transfer gain Δ = multitask − single-task, multi-seed, two comparison modes)
- **C2** — Efficiency benchmarking protocol (params, FLOPs, latency, peak memory, GPU energy)
- **C3** — Efficiency–performance Pareto frontier
- **C4** — Sequence-length scaling (32 → 512 tokens) under multitask training
- **C5** — Practical architecture-selection guidance (auto-generated from results)

**Tasks (all derived from CodeNet metadata)**
1. `problem_cls` — which of the 200 problems a submission solves
2. `verdict` — judge verdict (Accepted / Wrong Answer / TLE / Runtime Error / …)
3. `runtime_bucket` — tercile of `cpu_time` (a difficulty / complexity proxy)

**Methodological safeguards**

| Safeguard | Why it matters |
|---|---|
| Parameter-matched mode alongside default hyper-parameters | Separates the architecture effect from a raw capacity effect |
| 3 seeds per configuration (mean, 95% CI, Wilcoxon) | A single run cannot support claims about Δ |
| User-level (group-aware) train/val/test split | Stops a user's near-identical resubmissions leaking across splits |
| Near-duplicate removal (whitespace-normalised hash) | Removes trivially duplicated code before splitting |
| Macro-F1 + confusion matrices | Accuracy alone is misleading on the imbalanced `verdict` task |
| GPU energy logging (`pynvml`) | Energy is reported as a first-class efficiency metric |

**Before running:** set the `CODENET_PATH` environment variable (or edit `DATASET_PATH` in Section 0) to the
folder that contains CodeNet's `metadata/` and `data/` sub-folders. The full run trains 48 models for C1 plus
30 for C4; for a quick pipeline check set `n_seeds=1` and `comparison_modes=["default_hparams"]`.


In [ ]:
# =============================================================================
# 0. CONFIG
# =============================================================================
import os, sys, platform, random, json, time, glob, io, math, warnings, hashlib
from collections import Counter, defaultdict

# Reduce CUDA allocator fragmentation (must be set before torch grabs a CUDA context)
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# Root of your local Project CodeNet download. Set the CODENET_PATH environment
# variable or edit the fallback below. The folder must contain:
#   <DATASET_PATH>/metadata/p00000.csv, p00001.csv, ...
#   <DATASET_PATH>/data/p00000/Python/s000000000.py, ...
DATASET_PATH = os.environ.get("CODENET_PATH", r"path/to/Project_CodeNet")

CONFIG = dict(
    # --- data sampling ---
    n_problems=200,                    # top-N problems by Python submission count
    max_submissions_per_problem=150,
    min_submissions_per_problem=30,
    val_frac=0.15,
    test_frac=0.15,
    random_seed=42,                    # master seed (data sampling only; NOT the model seeds)
    seed_split=42,                     # seed for the user-level train/val/test split
    min_samples_per_split=10,
    dedup_near_identical=True,         # drop near-duplicate submissions before splitting

    # --- sequence lengths ---
    seq_len_grid=[32, 64, 128, 256, 512],  # C4 sweep (Section 10)
    seq_len_file=512,                      # "file-level" length for the main C1/C3 experiment

    # --- tokenizer / vocab ---
    max_vocab_size=50000,
    min_token_freq=5,
    max_code_tokens=2000,              # max tokens per file when pre-encoding (Section 2)
    sample_vocab_size=4000,            # number of files sampled to build the vocabulary

    # --- model dims (the "default_hparams" comparison mode) ---
    embed_dim=256,
    hidden_dim=128,                    # BiGRU hidden size / Transformer feed-forward size
    n_layers=3,
    n_heads=8,
    dropout=0.1,
    encoder_type="Transformer",        # encoder used for the forward-pass check in Section 4

    # --- training ---
    batch_size=64,
    max_epochs=15,                     # upper bound; early stopping decides actual epochs
    early_stop_patience=5,
    lr=3e-4,
    weight_decay=1e-5,
    device="cuda" if __import__("torch").cuda.is_available() else "cpu",

    # --- experimental rigour ---
    n_seeds=3,
    model_seeds=[1, 2, 3],             # len must equal n_seeds
    comparison_modes=["default_hparams", "param_matched"],
    enable_energy_logging=True,        # auto-disables if pynvml / GPU is unavailable
    balance_sampler_task="verdict",    # WeightedRandomSampler balances on this task's classes
    seq_len_n_seeds=3,                 # seeds averaged per point in the C4 sweep
)

random.seed(CONFIG["random_seed"])
np.random.seed(CONFIG["random_seed"])

import torch
torch.manual_seed(CONFIG["random_seed"])
print("Device:", CONFIG["device"])

# --- validate DATASET_PATH points at a real Project CodeNet root ---
_metadata_ok = os.path.isdir(os.path.join(DATASET_PATH, "metadata"))
_data_ok = os.path.isdir(os.path.join(DATASET_PATH, "data"))
print("DATASET_PATH:", DATASET_PATH)
print("  metadata/ found:", _metadata_ok)
print("  data/ found:    ", _data_ok)
if not (_metadata_ok and _data_ok):
    print("\n⚠️  DATASET_PATH does not look like a Project CodeNet root yet.")
    print("   Update the DATASET_PATH string above to the folder that directly contains")
    print("   'metadata' and 'data' subfolders, then re-run this cell.")

# --- runtime note ---
n_archs = 2
n_modes = len(CONFIG["comparison_modes"])
n_tasks_plus_mt = 4  # 3 single-task + 1 multitask
approx_runs_c1 = n_archs * n_modes * CONFIG["n_seeds"] * n_tasks_plus_mt
print(f"\nApprox. number of full training runs in the C1 experiment (Section 7): {approx_runs_c1}")
print("If that's too slow for a first pass, temporarily set comparison_modes=['default_hparams'] and/or "
      "n_seeds=1 in CONFIG above, verify the pipeline, then restore full settings for the real run.")


## 1. Index CodeNet metadata

CodeNet layout assumed:
```
Project_CodeNet/
  metadata/p00000.csv, p00001.csv, ...
  data/p00000/Python/s000000000.py, ...
  problem_list.csv
```
Indexing runs in two passes so we never have to load all ~14M submission rows into memory:
1. **Pass 1** (next cell) scans every `metadata/p*.csv` with only the `problem_id`/`language` columns to rank
   problems by Python-submission count, and picks the top `CONFIG["n_problems"]`.
2. **Pass 2** (cell after) reloads just those selected problems in full, resolves each submission's source-file
   path (`data/{problem_id}/Python/{submission_id}{filename_ext}`), drops rows whose file isn't on disk,
   deduplicates near-identical code, and builds the three task labels (`problem_cls`, `verdict`, `runtime_bucket`).

Result: a single `sample_df` with one row per usable submission, used by every section below.


In [ ]:
# =============================================================================
# 1a. LOCATE CODENET METADATA + RANK PROBLEMS BY PYTHON SUBMISSION COUNT (PASS 1)
# =============================================================================
import os
import glob
import time
import pandas as pd

METADATA_DIR = os.path.join(DATASET_PATH, "metadata")
DATA_DIR = os.path.join(DATASET_PATH, "data")
TARGET_LANGUAGE = "Python"   # CodeNet's mapped `language` column groups Python2/Python3 -> "Python"

if not os.path.isdir(METADATA_DIR) or not os.path.isdir(DATA_DIR):
    raise RuntimeError(
        f"Expected the Project CodeNet layout under DATASET_PATH:\n"
        f"  {DATASET_PATH}\\metadata\\p*.csv\n"
        f"  {DATASET_PATH}\\data\\pXXXXX\\<Language>\\sXXXXXXXXX.<ext>\n"
        f"metadata/ exists={os.path.isdir(METADATA_DIR)}, data/ exists={os.path.isdir(DATA_DIR)}.\n"
        f"Fix DATASET_PATH in Section 0 and re-run from there."
    )

meta_files = sorted(glob.glob(os.path.join(METADATA_DIR, "p*.csv")))
meta_files = [f for f in meta_files if os.path.basename(f).lower() != "problem_list.csv"]
print(f"Found {len(meta_files):,} per-problem metadata CSVs under {METADATA_DIR}")
if len(meta_files) == 0:
    raise RuntimeError(f"No p*.csv files found under {METADATA_DIR} — check DATASET_PATH.")

# --- cheap first pass: only read the 2 columns we need to rank problems ---
print(f"\nPass 1/2: counting '{TARGET_LANGUAGE}' submissions per problem "
      f"(scanning {len(meta_files):,} metadata files, 2 columns each)...")

problem_counts = {}
n_unreadable = 0
t0 = time.time()
for i, mf in enumerate(meta_files):
    try:
        dfi = pd.read_csv(mf, usecols=["problem_id", "language"], dtype=str)
    except Exception:
        n_unreadable += 1
        continue
    n = int((dfi["language"] == TARGET_LANGUAGE).sum())
    if n > 0:
        pid = os.path.splitext(os.path.basename(mf))[0]
        problem_counts[pid] = n
    if (i + 1) % 500 == 0:
        print(f"  scanned {i + 1:,}/{len(meta_files):,} files ({time.time() - t0:.0f}s elapsed)")

print(f"\nPass 1 done in {time.time() - t0:.0f}s.")
if n_unreadable:
    print(f"  ⚠️ {n_unreadable} metadata files could not be read and were skipped.")
print(f"  {len(problem_counts):,} problems have at least one {TARGET_LANGUAGE} submission.")

counts_series = pd.Series(problem_counts).sort_values(ascending=False)
eligible = counts_series[counts_series >= CONFIG["min_submissions_per_problem"]]
print(f"  {len(eligible):,} problems meet min_submissions_per_problem="
      f"{CONFIG['min_submissions_per_problem']}")

if CONFIG["n_problems"] is None:
    selected_problem_ids = list(eligible.index)
else:
    selected_problem_ids = list(eligible.head(CONFIG["n_problems"]).index)

print(f"\nSelected top {len(selected_problem_ids)} problems by {TARGET_LANGUAGE} submission count.")
print("\nTop 10 by count:")
print(eligible.head(10))


In [ ]:
# =============================================================================
# 1b. LOAD FULL METADATA FOR SELECTED PROBLEMS, RESOLVE CODE PATHS, BUILD TASKS (PASS 2)
# =============================================================================
import os
import time
import hashlib
import random
import pandas as pd

# Real Project CodeNet per-problem CSVs ship with NO header row; this is the
# fixed column order documented in the CodeNet README.
METADATA_COLS = ["submission_id", "problem_id", "user_id", "date", "language",
                  "original_language", "filename_ext", "status", "cpu_time",
                  "memory", "code_size", "accuracy"]

print(f"Pass 2/2: loading full metadata for {len(selected_problem_ids)} selected problems...")

rows = []
n_load_errors = 0
t0 = time.time()
for i, pid in enumerate(selected_problem_ids):
    mf = os.path.join(METADATA_DIR, f"{pid}.csv")
    try:
        dfi = pd.read_csv(mf, names=METADATA_COLS, header=0)
    except Exception as e:
        n_load_errors += 1
        continue
    dfi = dfi[dfi["language"] == TARGET_LANGUAGE].copy()
    if len(dfi) == 0:
        continue
    if CONFIG["max_submissions_per_problem"] is not None and len(dfi) > CONFIG["max_submissions_per_problem"]:
        dfi = dfi.sample(n=CONFIG["max_submissions_per_problem"], random_state=CONFIG["random_seed"])
    rows.append(dfi)
    if (i + 1) % 20 == 0:
        print(f"  loaded {i + 1}/{len(selected_problem_ids)} problems ({time.time() - t0:.0f}s elapsed)")

if n_load_errors:
    print(f"  ⚠️ {n_load_errors} selected metadata files failed to load and were skipped.")

if not rows:
    raise RuntimeError("No metadata rows loaded for any selected problem — check DATASET_PATH / Section 1a output.")

meta_df = pd.concat(rows, ignore_index=True)
print(f"\nLoaded {len(meta_df):,} {TARGET_LANGUAGE} submission records "
      f"across {meta_df['problem_id'].nunique()} problems.")

# --- resolve source-file paths: data/{problem_id}/{language}/{submission_id}{filename_ext} ---
def resolve_code_path(row):
    ext = row["filename_ext"]
    if not isinstance(ext, str) or ext == "" or ext.lower() == "nan":
        ext = ""
    elif not ext.startswith("."):
        ext = "." + ext
    return os.path.join(DATA_DIR, row["problem_id"], TARGET_LANGUAGE, f"{row['submission_id']}{ext}")

meta_df["code_path"] = meta_df.apply(resolve_code_path, axis=1)
meta_df["file_exists"] = meta_df["code_path"].apply(os.path.isfile)
n_missing = int((~meta_df["file_exists"]).sum())
if n_missing:
    print(f"⚠️  {n_missing:,} of {len(meta_df):,} rows point to a source file that isn't on disk "
          f"(partial download, or a different filename_ext convention) — dropping them.")
meta_df = meta_df[meta_df["file_exists"]].reset_index(drop=True)
print(f"Submissions with resolvable source files: {len(meta_df):,}")

if len(meta_df) == 0:
    raise RuntimeError(
        "No source files resolved on disk. Check that DATASET_PATH/data/ actually contains the "
        "downloaded .py source files (some partial CodeNet downloads ship metadata without data/)."
    )

# --- near-duplicate dedup (whitespace-normalized hash, per problem) ---
def normalized_hash(path):
    try:
        with open(path, "r", errors="ignore") as f:
            text = f.read()
        norm = "".join(text.split())
        return hashlib.md5(norm.encode("utf-8", errors="ignore")).hexdigest()
    except Exception:
        return None

if CONFIG["dedup_near_identical"]:
    print("\nDeduplicating near-identical submissions (whitespace-normalized hash)...")
    before = len(meta_df)
    meta_df["_code_hash"] = meta_df["code_path"].apply(normalized_hash)
    meta_df = meta_df.drop_duplicates(subset=["problem_id", "_code_hash"], keep="first").reset_index(drop=True)
    print(f"Dropped {before - len(meta_df):,} near-duplicates -> {len(meta_df):,} remain")

sample_df = meta_df.copy()

# --- Task 1: problem classification ---
problem_classes = sorted(sample_df["problem_id"].unique())
problem_to_idx = {p: i for i, p in enumerate(problem_classes)}
sample_df["problem_label"] = sample_df["problem_id"].map(problem_to_idx)

# --- Task 2: verdict classification ---
def normalize_status(s):
    if not isinstance(s, str):
        return "Other"
    s = s.strip()
    keep = {"Accepted", "Wrong Answer", "Time Limit Exceeded", "Runtime Error",
            "Compile Error", "Memory Limit Exceeded"}
    return s if s in keep else "Other"

sample_df["verdict_norm"] = sample_df["status"].apply(normalize_status)
verdict_classes = sorted(sample_df["verdict_norm"].unique())
verdict_to_idx = {v: i for i, v in enumerate(verdict_classes)}
sample_df["verdict_label"] = sample_df["verdict_norm"].map(verdict_to_idx)

print("\nVerdict class distribution:")
print(sample_df["verdict_norm"].value_counts(normalize=True).round(3))

# --- Task 3: runtime-bucket classification (cpu_time, falls back to code_size) ---
N_RUNTIME_BUCKETS = 3
cpu_time_numeric = pd.to_numeric(sample_df.get("cpu_time"), errors="coerce")
if cpu_time_numeric.notna().sum() > 0 and cpu_time_numeric.nunique(dropna=True) > 1:
    runtime_source_col = "cpu_time"
    vals = cpu_time_numeric.fillna(cpu_time_numeric.median())
else:
    runtime_source_col = "code_size"
    vals = pd.to_numeric(sample_df["code_size"], errors="coerce")
    # Fill NaNs here too -- qcut on ranked NaNs raises when casting to int below.
    if vals.notna().sum() > 0:
        vals = vals.fillna(vals.median())
    else:
        raise RuntimeError("Neither 'cpu_time' nor 'code_size' has usable numeric values "
                            "for the runtime_bucket task -- check the CodeNet metadata columns.")

sample_df["runtime_bucket_label"] = pd.qcut(
    vals.rank(method="first"), q=N_RUNTIME_BUCKETS, labels=False
).astype(int)
print(f"\nruntime_bucket task built from column: '{runtime_source_col}'")

# --- summary ---
print("\n" + "=" * 70)
print("✅ FINAL DATASET SUMMARY")
print("=" * 70)
print(f"Total samples: {len(sample_df):,}")
print(f"Total problems: {len(problem_classes)}")
print(f"Verdict classes: {len(verdict_classes)} -> {verdict_classes}")
print(f"Runtime buckets: {N_RUNTIME_BUCKETS} (from '{runtime_source_col}')")
print(f"Distinct users: {sample_df['user_id'].nunique():,} "
      f"(used for the leakage-safe group split in Section 3)")

print("\nFirst 5 rows:")
print(sample_df[["submission_id", "problem_id", "user_id", "status",
                  "code_size", "cpu_time", "verdict_norm"]].head())

# --- save for reuse (skip re-scanning the raw dataset on notebook re-runs) ---
sample_df.drop(columns=["file_exists"], errors="ignore").to_csv("sample_df_indexed.csv", index=False)
print("\n✅ Indexed dataset saved to 'sample_df_indexed.csv' "
      "(reload with pd.read_csv('sample_df_indexed.csv') to skip Section 1 on future runs)")


## 2. Tokenizer & vocabulary

Python's `tokenize` module turns source code into a lexical token stream (identifiers/strings/numbers generalized), avoiding a heavy external dependency.


In [ ]:
# =============================================================================
# 2. CODE TOKENIZER
# =============================================================================
import tokenize
import keyword
import re
from collections import Counter
import random

def tokenize_python_source(path, max_tokens=100000):
    """Lexical token stream of a Python source file.

    Identifiers become <ID>, string literals <STR>, numbers <NUM>; keywords and
    operators are kept verbatim; comments and layout tokens are dropped. Files
    the Python tokenizer rejects (e.g. Python 2 syntax) fall back to a simple
    regex split.
    """
    skip = (tokenize.ENCODING, tokenize.NEWLINE, tokenize.NL, tokenize.INDENT,
            tokenize.DEDENT, tokenize.ENDMARKER, tokenize.COMMENT)
    try:
        with open(path, "rb") as f:
            toks = []
            for tok in tokenize.tokenize(f.readline):
                if tok.type in skip:
                    continue
                s = tok.string
                if tok.type == tokenize.STRING:
                    s = "<STR>"
                elif tok.type == tokenize.NUMBER:
                    s = "<NUM>"
                elif tok.type == tokenize.NAME and s not in keyword.kwlist:
                    s = "<ID>"
                toks.append(s)
                if len(toks) >= max_tokens:
                    break
            return toks
    except Exception:
        try:
            with open(path, "r", errors="ignore") as f:
                text = f.read()
            return re.findall(r"\w+|[^\s\w]", text)[:max_tokens]
        except Exception:
            return []


PAD, UNK, CLS = "<PAD>", "<UNK>", "<CLS>"

def build_vocab(file_paths, max_vocab_size, min_freq, sample_cap=4000, verbose=True):
    """
    Build vocabulary from a sample of source files.
    """
    if verbose:
        print(f"\nBuilding vocabulary from {len(file_paths)} files...")
        print(f"Sampling up to {sample_cap} files...")

    # Sample files for vocabulary building
    if len(file_paths) <= sample_cap:
        sample_paths = file_paths
    else:
        sample_paths = random.sample(list(file_paths), sample_cap)

    if verbose:
        print(f"Tokenizing {len(sample_paths)} sampled files...")

    freq = Counter()
    processed = 0

    for i, p in enumerate(sample_paths):
        if (i + 1) % 500 == 0:
            print(f"  Tokenized {i + 1}/{len(sample_paths)} files...")

        # Check if file exists before tokenizing
        if os.path.exists(p):
            tokens = tokenize_python_source(p)
            freq.update(tokens)
            processed += 1

    if verbose:
        print(f"Successfully tokenized {processed} files")
        print(f"Found {len(freq)} unique tokens")

    # Filter by frequency and limit vocabulary size
    vocab_tokens = [t for t, c in freq.most_common() if c >= min_freq]
    vocab_tokens = vocab_tokens[:max_vocab_size - 3]  # Reserve space for special tokens

    itos = [PAD, UNK, CLS] + vocab_tokens
    stoi = {t: i for i, t in enumerate(itos)}

    if verbose:
        print(f"Vocabulary size: {len(itos)} (including {len(vocab_tokens)} tokens)")
        print(f"Most common tokens: {vocab_tokens[:10]}")

    return stoi, itos

def encode_code(path, max_len, vocab_stoi):
    """
    Encode a code file into token IDs.
    """
    # Get tokens from the file
    toks = tokenize_python_source(path, max_tokens=max_len - 1)  # Reserve space for CLS

    # Convert tokens to IDs
    ids = [vocab_stoi.get(tok, vocab_stoi[UNK]) for tok in toks]

    # Add CLS token at the beginning
    ids = [vocab_stoi[CLS]] + ids

    # Pad or truncate to max_len
    if len(ids) < max_len:
        ids = ids + [vocab_stoi[PAD]] * (max_len - len(ids))
    else:
        ids = ids[:max_len]

    return ids

# =============================================================================
# EXECUTION
# =============================================================================

print("=" * 70)
print("BUILDING CODE VOCABULARY AND ENCODING")
print("=" * 70)

# Check if we have files to process
if len(sample_df) == 0:
    raise RuntimeError("No files in sample_df. Run the indexing step first.")

print(f"\nProcessing {len(sample_df):,} files...")

# Get valid file paths
valid_paths = [p for p in sample_df["code_path"].tolist() if os.path.exists(p)]
print(f"Valid files: {len(valid_paths):,} out of {len(sample_df):,}")

if len(valid_paths) == 0:
    raise RuntimeError("No valid file paths found. Check your data.")

# Build vocabulary
print("\n" + "-" * 70)
vocab_stoi, vocab_itos = build_vocab(
    valid_paths,
    CONFIG["max_vocab_size"],
    CONFIG["min_token_freq"],
    sample_cap=CONFIG.get("sample_vocab_size", 4000),
    verbose=True
)
print("Vocabulary size:", len(vocab_itos))

# Sample code to show tokenization
print("\n" + "-" * 70)
print("TOKENIZATION EXAMPLE")
print("-" * 70)
sample_path = valid_paths[0]
print(f"Sample file: {os.path.basename(sample_path)}")
sample_tokens = tokenize_python_source(sample_path, max_tokens=50)
print(f"First 20 tokens: {sample_tokens[:20]}")
sample_encoding = encode_code(sample_path, 50, vocab_stoi)
print(f"Encoded tokens (first 20): {sample_encoding[:20]}")
print(f"Decoded tokens: {[vocab_itos[idx] for idx in sample_encoding[:20]]}")

# Encode all files in batches
print("\n" + "-" * 70)
print("ENCODING ALL FILES")
print("-" * 70)

# Encode in batches to manage memory
max_len = CONFIG.get("max_code_tokens", 2000)
batch_size = 1000  # Process 1000 files at a time

all_encodings = []
total_files = len(valid_paths)

print(f"Max sequence length: {max_len}")
print(f"Processing {total_files:,} files in batches of {batch_size}...")

for i in range(0, total_files, batch_size):
    batch_paths = valid_paths[i:i+batch_size]

    if (i + 1) % 5000 == 0 or i == 0:
        print(f"  Processing batch {i//batch_size + 1}/{(total_files//batch_size) + 1} "
              f"({min(i + batch_size, total_files):,}/{total_files:,} files)...")

    batch_encodings = []
    for path in batch_paths:
        encoding = encode_code(path, max_len, vocab_stoi)
        batch_encodings.append(encoding)

    all_encodings.extend(batch_encodings)

print(f"✅ Encoded all {total_files:,} files")

# Convert to numpy array
code_encodings = np.array(all_encodings, dtype=np.int32)
print(f"Encodings shape: {code_encodings.shape}")
print(f"Memory usage: {code_encodings.nbytes / 1024**2:.2f} MB")

# Add encodings to the dataframe
sample_df["code_encoding"] = list(code_encodings)
sample_df["code_length"] = [len([t for t in enc if t != vocab_stoi[PAD]]) for enc in code_encodings]

print("\n" + "-" * 70)
print("FINAL STATISTICS")
print("-" * 70)
print(f"Total files encoded: {len(sample_df):,}")
print(f"Average code length: {sample_df['code_length'].mean():.1f}")
print(f"Max code length: {sample_df['code_length'].max()}")
print(f"Min code length: {sample_df['code_length'].min()}")

# Show distribution of code lengths
print("\nCode length distribution:")
length_bins = [0, 50, 100, 200, 500, 1000, 2000, max_len]
for i in range(len(length_bins) - 1):
    count = ((sample_df['code_length'] >= length_bins[i]) &
             (sample_df['code_length'] < length_bins[i+1])).sum()
    if count > 0:
        print(f"  {length_bins[i]:4d} - {length_bins[i+1]:4d}: {count:5d} files ({count/len(sample_df)*100:.1f}%)")

print("\n✅ Tokenization and encoding complete!")

# Save encoded tokens + labels (~230 MB, git-ignored; not needed by later sections)
essential_with_encodings = sample_df[['submission_id', 'problem_id', 'problem_label',
                                       'verdict_label', 'runtime_bucket_label',
                                       'code_length', 'code_encoding']].copy()
# Save as pickle (more efficient for numpy arrays)
import pickle
with open('encoded_data.pkl', 'wb') as f:
    pickle.dump(essential_with_encodings, f)
print("\n✅ Encoded data saved to 'encoded_data.pkl'")


## 3. Dataset / leakage-safe split / task registry

**Group-aware split.** If `user_id` is available, we split by whole user-groups within each problem
(all of a user's submissions to a problem go entirely into train, val, or test) rather than splitting
individual rows — this prevents near-identical resubmissions from leaking across the split. Falls back
to plain per-row stratified splitting if `user_id` isn't populated in your CodeNet distribution.


In [ ]:
# =============================================================================
# 3. DATASETS + LEAKAGE-FREE SPLIT (MULTI-LEVEL STRATIFICATION)
# =============================================================================
import torch
from torch.utils.data import Dataset, DataLoader
import random
import numpy as np
from sklearn.model_selection import StratifiedShuffleSplit
from collections import Counter
import pandas as pd
import os

# --- TASK REGISTRY ---
TASK_LABEL_COLUMNS = {
    "problem_cls": "problem_label",
    "verdict": "verdict_label",
    "runtime_bucket": "runtime_bucket_label",
}
TASKS_ALL = list(TASK_LABEL_COLUMNS.keys())

# =============================================================================
# 3a. LEAKAGE-FREE SPLIT STRATEGIES
# =============================================================================

def group_aware_split_global(df, val_frac, test_frac, seed, group_col="user_id"):
    """
    Split users globally so each user appears in exactly one split.
    This prevents the same user from appearing in both training and validation/test sets.
    """
    rng = random.Random(seed)
    np.random.seed(seed)

    print(f"  Using GLOBAL group-aware split (each user in exactly one split)")

    # Make a copy to avoid modifying the original
    df = df.copy()

    # Handle missing user_ids - treat each as a unique user
    missing_mask = df[group_col].isna()
    n_missing = missing_mask.sum()

    if n_missing > 0:
        print(f"  ℹ️  Found {n_missing:,} rows with missing '{group_col}'")
        print(f"     Treating each missing row as its own unique user")
        # Create unique IDs for missing values
        df.loc[missing_mask, group_col] = [
            f"__missing_user_{i}" for i in range(n_missing)
        ]

    # Get unique users and their sample counts
    user_groups = df.groupby(group_col)
    user_counts = user_groups.size()
    n_users = len(user_counts)

    print(f"  Total unique users: {n_users:,}")

    # Shuffle users randomly
    users = user_counts.index.tolist()
    rng.shuffle(users)

    # Calculate target numbers of users for each split
    n_val_users = max(1, int(n_users * val_frac))
    n_test_users = max(1, int(n_users * test_frac))

    # Handle edge cases where we don't have enough users
    if n_val_users + n_test_users >= n_users:
        # Not enough users for all splits, adjust
        n_val_users = max(1, int(n_users * val_frac / (val_frac + test_frac)))
        n_test_users = n_users - n_val_users - 1  # Leave at least 1 for training

        if n_test_users < 1:
            n_test_users = 1
            n_val_users = n_users - 2

        if n_val_users < 1:
            n_val_users = 1
            n_test_users = n_users - 2

    print(f"  Split user counts: Train={n_users - n_val_users - n_test_users:,}, "
          f"Val={n_val_users:,}, Test={n_test_users:,}")

    # Assign users to splits
    val_users = set(users[:n_val_users])
    test_users = set(users[n_val_users:n_val_users + n_test_users])
    train_users = set(users[n_val_users + n_test_users:])

    # Verify no overlap between splits
    assert len(val_users.intersection(test_users)) == 0, "Val and Test users overlap!"
    assert len(train_users.intersection(val_users)) == 0, "Train and Val users overlap!"
    assert len(train_users.intersection(test_users)) == 0, "Train and Test users overlap!"

    # Assign all submissions of each user to their split
    train_df = df[df[group_col].isin(train_users)].copy()
    val_df = df[df[group_col].isin(val_users)].copy()
    test_df = df[df[group_col].isin(test_users)].copy()

    # Reset indices
    train_df.reset_index(drop=True, inplace=True)
    val_df.reset_index(drop=True, inplace=True)
    test_df.reset_index(drop=True, inplace=True)

    # Print verification statistics
    print(f"\n  Split sample counts:")
    print(f"    Train: {len(train_df):,} samples ({len(train_df)/len(df)*100:.1f}%)")
    print(f"    Val:   {len(val_df):,} samples ({len(val_df)/len(df)*100:.1f}%)")
    print(f"    Test:  {len(test_df):,} samples ({len(test_df)/len(df)*100:.1f}%)")

    return train_df, val_df, test_df

def stratified_split(df, val_frac, test_frac, seed, stratify_col, min_samples_per_class=10):
    """
    Stratified split by verdict to maintain class distribution.
    Uses multi-level stratification for better balance.
    """
    print(f"  Stratifying by: {stratify_col}")

    # Check class distribution
    class_counts = df[stratify_col].value_counts()
    print(f"  Class distribution in full dataset:")
    for cls, count in class_counts.items():
        print(f"    {cls}: {count:,} ({count/len(df)*100:.1f}%)")

    # Ensure minimum samples per class in each split
    small_classes = class_counts[class_counts < min_samples_per_class * 3].index.tolist()
    if small_classes:
        print(f"   Small classes detected: {small_classes}")
        print(f"  These may end up under-represented (or missing) in some splits; "
              f"StratifiedShuffleSplit below does its best but cannot guarantee "
              f"min_samples_per_class for very rare classes.")

    # First, split into train+val vs test
    train_val_size = 1 - test_frac
    sss1 = StratifiedShuffleSplit(n_splits=1, test_size=test_frac, random_state=seed)

    try:
        train_val_idx, test_idx = next(sss1.split(df, df[stratify_col]))
    except ValueError as e:
        print(f"    Stratified split failed: {e}")
        print(f"  Falling back to random split")
        # Fallback to random split
        indices = np.random.permutation(len(df))
        test_idx = indices[:int(len(df) * test_frac)]
        train_val_idx = indices[int(len(df) * test_frac):]

    train_val_df = df.iloc[train_val_idx].reset_index(drop=True)
    test_df = df.iloc[test_idx].reset_index(drop=True)

    # Then split train_val into train and val
    val_frac_adjusted = val_frac / (1 - test_frac) if (1 - test_frac) > 0 else 0.5

    sss2 = StratifiedShuffleSplit(n_splits=1, test_size=val_frac_adjusted, random_state=seed)

    try:
        train_idx, val_idx = next(sss2.split(train_val_df, train_val_df[stratify_col]))
    except ValueError:
        # Fallback to random split
        indices = np.random.permutation(len(train_val_df))
        val_idx = indices[:int(len(train_val_df) * val_frac_adjusted)]
        train_idx = indices[int(len(train_val_df) * val_frac_adjusted):]

    train_df = train_val_df.iloc[train_idx].reset_index(drop=True)
    val_df = train_val_df.iloc[val_idx].reset_index(drop=True)

    print(f"  Split sizes: Train={len(train_df):,}, Val={len(val_df):,}, Test={len(test_df):,}")

    return train_df, val_df, test_df

# =============================================================================
# 3b. CUSTOM DATASET CLASS
# =============================================================================

class CodeNetMultiTaskDataset(Dataset):
    """
    Dataset class for multi-task learning on CodeNet data.
    Handles encoding and label extraction on-the-fly.
    """
    def __init__(self, df, max_len, vocab_stoi, use_cache=True):
        self.paths = df["code_path"].tolist()
        self.label_columns = {task: df[col].tolist() for task, col in TASK_LABEL_COLUMNS.items()}
        self.max_len = max_len
        self.vocab_stoi = vocab_stoi
        self.use_cache = use_cache
        self._cache = {} if use_cache else None

        print(f"Dataset initialized with {len(self.paths):,} samples")
        print(f"  Tasks: {list(self.label_columns.keys())}")

        # Verify all paths exist
        missing = sum(1 for p in self.paths if not os.path.exists(p))
        if missing > 0:
            print(f"  ⚠️  Warning: {missing} files are missing")

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        path = self.paths[idx]
        key = (path, self.max_len)

        # Get encoding (with caching)
        if self.use_cache and key in self._cache:
            ids = self._cache[key]
        else:
            ids = encode_code(path, self.max_len, self.vocab_stoi)
            if self.use_cache:
                self._cache[key] = ids

        # Get labels for all tasks
        labels = {}
        for task, vals in self.label_columns.items():
            labels[task] = torch.tensor(vals[idx], dtype=torch.long)

        return torch.tensor(ids, dtype=torch.long), labels

# =============================================================================
# 3c. DATALOADER CREATION WITH BALANCED SAMPLING
# =============================================================================

def create_balanced_sampler(labels, num_classes):
    """
    Create a weighted sampler for imbalanced datasets.
    """
    class_counts = Counter(labels)
    weights = []
    for label in labels:
        weights.append(1.0 / class_counts[label])

    sampler = torch.utils.data.WeightedRandomSampler(
        weights,
        num_samples=len(weights),
        replacement=True
    )
    return sampler

def make_loaders(train_df, val_df, test_df, max_len, vocab_stoi,
                 batch_size=None, use_balanced_sampler=False, eval_batch_size=None):
    """
    Create DataLoaders for training, validation, and testing.

    `eval_batch_size` is deliberately smaller than the training batch size by
    default. Val/test passes don't need gradients, but the Transformer's
    self-attention still allocates an L x L score matrix per head per layer
    at *whatever* batch size you give it -- that memory scales with
    batch_size * seq_len^2, so a large eval batch can OOM even when the
    same-sized training batch (with the same model) was completely fine.
    """
    bs = batch_size or CONFIG.get("batch_size", 64)
    eval_bs = eval_batch_size or CONFIG.get("eval_batch_size", max(4, bs // 4))

    print(f"\nCreating DataLoaders with batch_size={bs} (train), {eval_bs} (val/test)")

    # Create datasets
    train_ds = CodeNetMultiTaskDataset(train_df, max_len, vocab_stoi)
    val_ds = CodeNetMultiTaskDataset(val_df, max_len, vocab_stoi)
    test_ds = CodeNetMultiTaskDataset(test_df, max_len, vocab_stoi)

    # Create samplers
    train_sampler = None
    if use_balanced_sampler and len(train_df) > 0:
        # Balance by the task that's actually imbalanced (verdict is ~55% Accepted,
        # per the methodology notes) -- configurable via CONFIG["balance_sampler_task"]
        # rather than silently taking "first task in the dict", which used to resolve
        # to problem_cls and balance the wrong thing.
        balance_task = CONFIG.get("balance_sampler_task", "verdict")
        if balance_task not in TASK_LABEL_COLUMNS:
            print(f"    balance_sampler_task='{balance_task}' not in TASK_LABEL_COLUMNS, "
                  f"falling back to '{list(TASK_LABEL_COLUMNS.keys())[0]}'")
            balance_task = list(TASK_LABEL_COLUMNS.keys())[0]
        labels = train_df[TASK_LABEL_COLUMNS[balance_task]].tolist()
        num_classes = len(set(labels))
        train_sampler = create_balanced_sampler(labels, num_classes)
        print(f"  Using balanced sampler for training (balancing on '{balance_task}')")

    # Create dataloaders
    train_loader = DataLoader(
        train_ds,
        batch_size=bs,
        shuffle=(train_sampler is None),
        sampler=train_sampler,
        num_workers=0  # Set to 0 for Windows, increase for Linux/Mac
    )

    val_loader = DataLoader(
        val_ds,
        batch_size=eval_bs,
        shuffle=False,
        num_workers=0
    )

    test_loader = DataLoader(
        test_ds,
        batch_size=eval_bs,
        shuffle=False,
        num_workers=0
    )

    print(f"  Train batches: {len(train_loader)}")
    print(f"  Val batches: {len(val_loader)}")
    print(f"  Test batches: {len(test_loader)}")

    return train_loader, val_loader, test_loader

# =============================================================================
# 3d. EXECUTION
# =============================================================================

print("=" * 70)
print("CREATING LEAKAGE-FREE SPLITS")
print("=" * 70)

# First, check if we have sample_df
if 'sample_df' not in locals() or len(sample_df) == 0:
    raise RuntimeError("sample_df is empty or not defined. Run indexing and tokenization first.")

print(f"Total samples available: {len(sample_df):,}")

# Check for user_id
has_user_id = "user_id" in sample_df.columns and sample_df["user_id"].notna().any()
print(f"Has user_id: {has_user_id}")

# Create splits based on available data
if has_user_id:
    print("\nUsing global group-aware split (leakage-free by user_id)")
    train_df, val_df, test_df = group_aware_split_global(
        sample_df,
        CONFIG["val_frac"],
        CONFIG["test_frac"],
        CONFIG["seed_split"],
        "user_id"
    )
else:
    print("\nNo user_id found. Using stratified split by verdict")
    # Use verdict for stratification
    if "verdict_label" in sample_df.columns:
        stratify_col = "verdict_label"
    else:
        stratify_col = "problem_label"

    train_df, val_df, test_df = stratified_split(
        sample_df,
        CONFIG["val_frac"],
        CONFIG["test_frac"],
        CONFIG["seed_split"],
        stratify_col
    )

# Verify that no user appears in more than one split
if has_user_id and "user_id" in train_df.columns:
    # Get all user IDs from each split
    train_users = set(train_df["user_id"].dropna().astype(str))
    val_users = set(val_df["user_id"].dropna().astype(str))
    test_users = set(test_df["user_id"].dropna().astype(str))

    leakage_train_val = train_users.intersection(val_users)
    leakage_train_test = train_users.intersection(test_users)
    leakage_val_test = val_users.intersection(test_users)

    print(f"\n🔍 LEAKAGE VERIFICATION:")
    print(f"  Train users: {len(train_users):,}")
    print(f"  Val users:   {len(val_users):,}")
    print(f"  Test users:  {len(test_users):,}")
    print(f"  Train-Val overlap: {len(leakage_train_val)} users")
    print(f"  Train-Test overlap: {len(leakage_train_test)} users")
    print(f"  Val-Test overlap: {len(leakage_val_test)} users")

    if len(leakage_train_val) == 0 and len(leakage_train_test) == 0 and len(leakage_val_test) == 0:
        print("   NO LEAKAGE DETECTED! All users are in exactly one split.")
    else:
        print("   WARNING: Leakage detected! Users appear in multiple splits.")

# Display split statistics
print("\n" + "-" * 70)
print("SPLIT STATISTICS")
print("-" * 70)
print(f"Train: {len(train_df):,} samples ({len(train_df)/len(sample_df)*100:.1f}%)")
print(f"Val:   {len(val_df):,} samples ({len(val_df)/len(sample_df)*100:.1f}%)")
print(f"Test:  {len(test_df):,} samples ({len(test_df)/len(sample_df)*100:.1f}%)")

# Check class distribution in each split
if "verdict_label" in sample_df.columns:
    print("\nVerdict distribution across splits:")
    for split_name, split_df in [("Train", train_df), ("Val", val_df), ("Test", test_df)]:
        if len(split_df) > 0:
            dist = split_df["verdict_label"].value_counts(normalize=True).round(3)
            print(f"  {split_name}: {dist.to_dict()}")

# Create DataLoaders
print("\n" + "-" * 70)
print("CREATING DATALOADERS")
print("-" * 70)

# Get max sequence length from config.
max_len = CONFIG.get("seq_len_file", 512)

train_loader, val_loader, test_loader = make_loaders(
    train_df,
    val_df,
    test_df,
    max_len,
    vocab_stoi,
    batch_size=CONFIG.get("batch_size", 64),
    use_balanced_sampler=True,  # Helps with imbalanced verdict classes
    eval_batch_size=CONFIG.get("eval_batch_size", 16),
)

# Calculate task head sizes
N_PROBLEM_CLASSES = len(sample_df["problem_label"].unique())
N_VERDICT_CLASSES = len(sample_df["verdict_label"].unique())
N_RUNTIME_BUCKETS = len(sample_df["runtime_bucket_label"].unique())
VOCAB_SIZE = len(vocab_itos)
PAD_IDX = vocab_stoi[PAD]

task_head_sizes_all = {
    "problem_cls": N_PROBLEM_CLASSES,
    "verdict": N_VERDICT_CLASSES,
    "runtime_bucket": N_RUNTIME_BUCKETS,
}

print("\n" + "-" * 70)
print("FINAL CONFIGURATION")
print("-" * 70)
print(f"N_PROBLEM_CLASSES: {N_PROBLEM_CLASSES}")
print(f"N_VERDICT_CLASSES: {N_VERDICT_CLASSES}")
print(f"N_RUNTIME_BUCKETS: {N_RUNTIME_BUCKETS}")
print(f"VOCAB_SIZE: {VOCAB_SIZE}")
print(f"PAD_IDX: {PAD_IDX}")
print(f"Tasks: {TASKS_ALL}")
print(f"Task head sizes: {task_head_sizes_all}")

print("\n Dataset preparation complete!")
print(f"Ready for training with {len(train_df):,} training samples")


## 3b. Baseline (non-neural): TF-IDF + Logistic Regression

Before training any neural architecture, we establish a standard, cheap,
non-sequential baseline on the same `train_df`/`val_df`/`test_df` splits and
the same source-code tokenization used everywhere else in this notebook.
This answers the question a reviewer will ask first: *do the neural
architectures actually add value over a simple bag-of-tokens classifier?*

This baseline is single-task only by construction (TF-IDF + Logistic
Regression has no natural multitask analog) — it is a reference point for
the single-task accuracy numbers, not part of the multitask (C1) comparison
itself. It runs on CPU in seconds, so it's a fast sanity check to run before
committing to the much longer neural training loop in Section 7.


In [ ]:
# =============================================================================
# 3b. BASELINE: TF-IDF + LOGISTIC REGRESSION
# =============================================================================
import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.model_selection import GridSearchCV

def tokens_for_baseline(path, max_tokens=2000):
    """
    Raw tokenization for baseline models.
    No special tokens, no vocabulary mapping - just raw tokens from source.
    """
    try:
        toks = tokenize_python_source(path, max_tokens=max_tokens)
        special_tokens = ['<PAD>', '<UNK>', '<CLS>', '<SEP>', '<MASK>']
        return [t for t in toks if t not in special_tokens]
    except Exception as e:
        print(f"Warning: Failed to tokenize {path}: {e}")
        return []

def run_tfidf_logreg_baseline(train_df, val_df, test_df, task_label_columns,
                               tune_hyperparams=True, verbose=True):
    """
    Run TF-IDF + Logistic Regression baseline with optional hyperparameter tuning.
    """
    results = []

    for task, label_col in task_label_columns.items():
        if verbose:
            print(f"\n[baseline] {task}: tokenizing + TF-IDF + LogisticRegression ...")

        # Tokenize all splits
        train_tokens = [tokens_for_baseline(p) for p in train_df["code_path"]]
        val_tokens = [tokens_for_baseline(p) for p in val_df["code_path"]]
        test_tokens = [tokens_for_baseline(p) for p in test_df["code_path"]]

        # Get labels
        y_train = train_df[label_col].tolist()
        y_val = val_df[label_col].tolist()
        y_test = test_df[label_col].tolist()

        # TF-IDF Vectorization
        vec = TfidfVectorizer(
            analyzer=lambda toks: toks,
            min_df=2,  # Ignore tokens appearing in <2 documents
            max_df=0.95,  # Ignore tokens appearing in >95% of documents
            ngram_range=(1, 2),  # Include bigrams for better features
            max_features=10000  # Limit feature space for memory efficiency
        )

        Xtr = vec.fit_transform(train_tokens)
        Xval = vec.transform(val_tokens)
        Xte = vec.transform(test_tokens)

        if verbose:
            print(f"  Feature matrix: {Xtr.shape[1]:,} features, {Xtr.shape[0]:,} train samples")

        # Hyperparameter tuning (optional)
        fit_t0 = time.perf_counter()
        if tune_hyperparams:
            if verbose:
                print("  Performing hyperparameter tuning...")

            param_grid = {
                'C': [0.1, 1.0, 10.0],
                'solver': ['liblinear', 'saga'],
                'penalty': ['l1', 'l2']
            }

            # 3-fold CV on the training split
            clf = GridSearchCV(
                LogisticRegression(max_iter=2000, random_state=42),
                param_grid,
                cv=3,
                scoring='accuracy',
                n_jobs=-1
            )
            clf.fit(Xtr, y_train)
            best_clf = clf.best_estimator_

            if verbose:
                print(f"  Best params: {clf.best_params_}")
                print(f"  Best CV score: {clf.best_score_:.4f}")
        else:
            # Use default parameters
            best_clf = LogisticRegression(
                C=1.0,
                solver='liblinear',
                penalty='l2',
                max_iter=2000,
                random_state=42
            )
            best_clf.fit(Xtr, y_train)

        fit_time = time.perf_counter() - fit_t0

        # Evaluate on test set
        start = time.perf_counter()
        preds = best_clf.predict(Xte)
        pred_time = time.perf_counter() - start

        # Calculate metrics
        acc = accuracy_score(y_test, preds)
        f1_macro = f1_score(y_test, preds, average="macro")
        f1_weighted = f1_score(y_test, preds, average="weighted")

        # Baseline: majority class prediction
        majority_label = max(set(y_test), key=y_test.count)
        majority_baseline = y_test.count(majority_label) / len(y_test)

        # Store results
        results.append({
            "arch": "TFIDF+LogReg",
            "mode": "baseline",
            "task": task,
            "accuracy": round(acc, 4),
            "macro_f1": round(f1_macro, 4),
            "weighted_f1": round(f1_weighted, 4),
            "majority_baseline": round(majority_baseline, 4),
            "lift_over_majority": round(acc - majority_baseline, 4),
            "n_features": Xtr.shape[1],
            "pred_time_s": round(pred_time, 3),
            "train_time_s": round(fit_time, 2),
            "n_train_samples": len(train_df),
            "n_test_samples": len(test_df)
        })

        if verbose:
            print(f"  Test accuracy: {acc:.4f}")
            print(f"  Macro F1: {f1_macro:.4f}")
            print(f"  Weighted F1: {f1_weighted:.4f}")
            print(f"  Majority baseline: {majority_baseline:.4f}")
            print(f"  Lift over majority: {acc - majority_baseline:+.4f}")
            print(f"  Prediction time: {pred_time:.3f}s")

        # Print classification report for detailed performance
        if verbose:
            print(f"\n  Classification Report ({task}):")
            print(classification_report(y_test, preds))

    return pd.DataFrame(results)

# =============================================================================
# EXECUTION
# =============================================================================

print("=" * 70)
print("RUNNING NON-NEURAL BASELINE")
print("=" * 70)

# Run baseline with hyperparameter tuning
baseline_df = run_tfidf_logreg_baseline(
    train_df=train_df,
    val_df=val_df,
    test_df=test_df,
    task_label_columns=TASK_LABEL_COLUMNS,
    tune_hyperparams=True,
    verbose=True
)

# Save results
baseline_df.to_csv("baseline_results.csv", index=False)
print("\n" + "=" * 70)
print("BASELINE RESULTS (saved to baseline_results.csv)")
print("=" * 70)
print(baseline_df.to_string(index=False))

# =============================================================================
# VISUALIZATION
# =============================================================================

# Plot 1: Baseline vs Majority Class
fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(len(baseline_df))
width = 0.35

bars1 = ax.bar(x - width/2, baseline_df["majority_baseline"], width,
               label="Majority-class (chance)", color="#B0B0B0")
bars2 = ax.bar(x + width/2, baseline_df["accuracy"], width,
               label="TF-IDF + LogReg", color="#8858C8")

ax.set_xticks(x)
ax.set_xticklabels(baseline_df["task"])
ax.set_ylabel("Test Accuracy")
ax.set_title("Non-neural Baseline vs. Chance, per Task")
ax.legend()
ax.grid(axis="y", alpha=0.3)

# Add value labels on bars
for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax.annotate(f'{height:.3f}',
                   xy=(bar.get_x() + bar.get_width() / 2, height),
                   xytext=(0, 3),
                   textcoords="offset points",
                   ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.savefig("baseline_vs_majority.png", dpi=120)
print("\nPlot saved: baseline_vs_majority.png")
plt.show()

# Plot 2: Lift over majority baseline
fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(len(baseline_df))
bars = ax.bar(x, baseline_df["lift_over_majority"], color="#8858C8")
ax.set_xticks(x)
ax.set_xticklabels(baseline_df["task"])
ax.set_ylabel("Accuracy Lift over Majority")
ax.set_title("Improvement over Random Chance Baseline")
ax.axhline(y=0, color='red', linestyle='--', alpha=0.5)
ax.grid(axis="y", alpha=0.3)

# Add value labels
for bar in bars:
    height = bar.get_height()
    ax.annotate(f'{height:+.3f}',
               xy=(bar.get_x() + bar.get_width() / 2, height),
               xytext=(0, 3),
               textcoords="offset points",
               ha='center', va='bottom' if height > 0 else 'top', fontsize=8)

plt.tight_layout()
plt.savefig("baseline_lift.png", dpi=120)
print("Plot saved: baseline_lift.png")
plt.show()

print("\n✅ Baseline complete!")
print(f"Results saved to baseline_results.csv")
print(f"Plots saved to baseline_vs_majority.png and baseline_lift.png")


## 4. Model architectures

Each encoder maps `input_ids -> pooled representation`. `hidden_dim` drives BiGRU's recurrent width and
Transformer's feed-forward width — both are varied together in Section 4b to build the parameter-matched condition.


In [ ]:
# =============================================================================
# 4. ENCODERS: BiGRU / Transformer (PARAMETER-MATCHED ~1M params)
# =============================================================================
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import os
import numpy as np

# =============================================================================
# 4a. ENCODER MODELS (PARAMETER-MATCHED ~1M)
# =============================================================================

class BiGRUEncoder(nn.Module):
    """Bidirectional GRU encoder with ~1M parameters."""
    ARCH_NAME = "BiGRU"

    def __init__(self, vocab_size, embed_dim, hidden_dim, n_layers, dropout, pad_idx):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)

        # Calculate GRU parameters for ~1M total (encoder body, excluding the shared
        # input embedding table -- see Section 4b's parameter-matched calibration).
        # For ~1M: embed_dim=256, hidden_dim=128, n_layers=3 gives ~0.93M total params
        # (~0.89M encoder body). This is the value actually used below via CONFIG.
        self.gru = nn.GRU(
            embed_dim,
            hidden_dim,
            num_layers=n_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if n_layers > 1 else 0.0
        )
        self.out_dim = hidden_dim * 2
        self.norm = nn.LayerNorm(self.out_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, pad_mask):
        e = self.embed(x)
        e = self.dropout(e)

        lengths = pad_mask.sum(dim=1).cpu().int()
        packed = nn.utils.rnn.pack_padded_sequence(
            e, lengths, batch_first=True, enforce_sorted=False
        )
        out_packed, _ = self.gru(packed)
        out, _ = nn.utils.rnn.pad_packed_sequence(
            out_packed, batch_first=True, total_length=x.size(1)
        )

        lengths = pad_mask.sum(dim=1, keepdim=True).clamp(min=1)
        pooled = (out * pad_mask.unsqueeze(-1)).sum(dim=1) / lengths

        return self.norm(pooled)


class PositionalEncoding(nn.Module):
    """Sinusoidal positional encoding."""

    def __init__(self, d_model, max_len=2048):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]


class TransformerEncoderModel(nn.Module):
    """Transformer encoder with ~1M parameters."""
    ARCH_NAME = "Transformer"

    def __init__(self, vocab_size, embed_dim, dim_feedforward, n_layers, n_heads, dropout, pad_idx):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.pos = PositionalEncoding(embed_dim)
        self.dropout = nn.Dropout(dropout)

        # Calculate Transformer parameters for ~1M
        # Total params = vocab_size * embed_dim + n_layers * (embed_dim^2 * 4 + embed_dim * dim_feedforward * 2)
        # For ~1M: embed_dim=256, dim_feedforward=512, n_layers=3, n_heads=8 gives ~1.0M params
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=n_heads,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True,
            activation='gelu'
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.out_dim = embed_dim
        self.norm = nn.LayerNorm(embed_dim)

    def forward(self, x, pad_mask):
        e = self.embed(x)
        e = self.dropout(e)
        e = self.pos(e)

        key_padding_mask = (pad_mask == 0)
        out = self.encoder(e, src_key_padding_mask=key_padding_mask)

        lengths = pad_mask.sum(dim=1, keepdim=True).clamp(min=1)
        pooled = (out * pad_mask.unsqueeze(-1)).sum(dim=1) / lengths

        return self.norm(pooled)


# =============================================================================
# Architectures compared throughout the notebook
# =============================================================================
ARCHITECTURES = ["BiGRU", "Transformer"]


# =============================================================================
# 4c. ENCODER FACTORY WITH ~1M PARAMETER MATCHING
# =============================================================================

def get_encoder_parameter_count(encoder):
    """Get total parameter count of an encoder."""
    return sum(p.numel() for p in encoder.parameters() if p.requires_grad)

def get_encoder_configs(encoder_name, vocab_size, pad_idx):
    """
    Get parameter-matched configurations for each encoder type.
    All configured to have approximately 1M parameters.
    """
    configs = {
        "BiGRU": {
            "embed_dim": 256,
            "hidden_dim": 128,
            "n_layers": 3,
            "dropout": 0.1,
        },
        "Transformer": {
            "embed_dim": 256,
            "dim_feedforward": 512,
            "n_layers": 3,
            "n_heads": 8,
            "dropout": 0.1,
        },
    }
    return configs[encoder_name]

def build_encoder_matched(name, vocab_size, pad_idx):
    """
    Build an encoder with approximately 1M parameters for fair comparison.

    Args:
        name: "BiGRU" or "Transformer"
        vocab_size: Size of vocabulary
        pad_idx: Padding token index

    Returns:
        Encoder module with ~1M parameters
    """
    config = get_encoder_configs(name, vocab_size, pad_idx)
    c = CONFIG

    print(f"\nBuilding {name} with ~1M parameters:")

    if name == "BiGRU":
        encoder = BiGRUEncoder(
            vocab_size,
            config["embed_dim"],
            config["hidden_dim"],
            config["n_layers"],
            config["dropout"],
            pad_idx
        )
        print(f"  embed_dim={config['embed_dim']}, hidden_dim={config['hidden_dim']}, "
              f"n_layers={config['n_layers']}")

    elif name == "Transformer":
        encoder = TransformerEncoderModel(
            vocab_size,
            config["embed_dim"],
            config["dim_feedforward"],
            config["n_layers"],
            config["n_heads"],
            config["dropout"],
            pad_idx
        )
        print(f"  embed_dim={config['embed_dim']}, dim_feedforward={config['dim_feedforward']}, "
              f"n_layers={config['n_layers']}, n_heads={config['n_heads']}")

    else:
        raise ValueError(f"Unknown encoder architecture: {name}")

    # Get parameter count
    params = get_encoder_parameter_count(encoder)
    print(f"  Total parameters: {params:,}")

    return encoder

def build_all_encoders(vocab_size, pad_idx):
    """
    Build both encoders (BiGRU, Transformer) and print their sizes.

    Returns:
        Dictionary of encoders
    """
    encoders = {}
    architectures = ["BiGRU", "Transformer"]

    print("\n" + "=" * 70)
    print("BUILDING PARAMETER-MATCHED ENCODERS (~1M params each)")
    print("=" * 70)

    for arch in architectures:
        encoder = build_encoder_matched(arch, vocab_size, pad_idx)
        encoders[arch] = encoder

    print("\n" + "-" * 70)
    print("PARAMETER SUMMARY:")
    print("-" * 70)
    for arch, encoder in encoders.items():
        params = get_encoder_parameter_count(encoder)
        print(f"  {arch:12s}: {params:>10,} parameters")

    return encoders

# =============================================================================
# 4d. MULTI-TASK MODEL
# =============================================================================

class MultiTaskModel(nn.Module):
    """
    Multi-task model with shared encoder and task-specific heads.
    """

    def __init__(self, encoder, task_head_sizes: dict):
        super().__init__()
        self.encoder = encoder
        self.heads = nn.ModuleDict({
            name: nn.Linear(encoder.out_dim, n_classes)
            for name, n_classes in task_head_sizes.items()
        })
        self.task_weights = {name: 1.0 for name in task_head_sizes.keys()}

        print(f"\nMultiTaskModel initialized:")
        print(f"  Encoder: {encoder.ARCH_NAME}")
        print(f"  Encoder output dim: {encoder.out_dim}")
        print(f"  Tasks: {list(task_head_sizes.keys())}")
        for name, n_classes in task_head_sizes.items():
            print(f"    {name}: {n_classes} classes")

    def forward(self, x, pad_mask):
        """
        Forward pass through the model.

        Args:
            x: Token indices (batch, seq_len)
            pad_mask: Padding mask (batch, seq_len)

        Returns:
            Dictionary of task outputs
        """
        rep = self.encoder(x, pad_mask)

        outputs = {}
        for name, head in self.heads.items():
            outputs[name] = head(rep)

        return outputs

    def forward_with_loss(self, x, pad_mask, labels):
        """
        Forward pass with loss computation.

        Args:
            x: Token indices (batch, seq_len)
            pad_mask: Padding mask (batch, seq_len)
            labels: Dictionary of task labels

        Returns:
            outputs: Dictionary of task outputs
            losses: Dictionary of task losses
            total_loss: Weighted sum of all losses
        """
        outputs = self.forward(x, pad_mask)

        losses = {}
        total_loss = torch.tensor(0.0, device=x.device)

        for task_name, logits in outputs.items():
            if task_name in labels:
                loss = F.cross_entropy(logits, labels[task_name])
                losses[task_name] = loss
                total_loss = total_loss + self.task_weights.get(task_name, 1.0) * loss

        return outputs, losses, total_loss


def make_pad_mask(x, pad_idx):
    """Create a padding mask from token indices."""
    return (x != pad_idx).float()


# =============================================================================
# 4e. EXECUTION
# =============================================================================

print("\n" + "=" * 70)
print("ENCODER INITIALIZATION (~1M PARAMETERS EACH)")
print("=" * 70)

# Check if required variables exist
if 'vocab_stoi' not in locals() or 'vocab_itos' not in locals():
    raise RuntimeError("Vocabulary not built. Run tokenization first.")

if 'task_head_sizes_all' not in locals():
    raise RuntimeError("Task head sizes not defined. Run dataset preparation first.")

# Get PAD index
PAD_IDX = vocab_stoi.get(PAD, 0)

# Build all encoders for comparison
all_encoders = build_all_encoders(VOCAB_SIZE, PAD_IDX)

# Build a specific encoder for use (change this to test different architectures)
selected_encoder_name = CONFIG.get("encoder_type", "Transformer")
selected_encoder = all_encoders[selected_encoder_name]

print(f"\nSelected encoder for use: {selected_encoder_name}")

# Build multi-task model with selected encoder
model = MultiTaskModel(selected_encoder, task_head_sizes_all)

print(f"\n✅ Model built successfully!")
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total model parameters: {total_params:,}")

# Test forward pass with dummy data
print("\nTesting forward pass with dummy data...")
dummy_batch = torch.randint(0, VOCAB_SIZE, (4, CONFIG.get("max_code_tokens", 100)))
dummy_mask = make_pad_mask(dummy_batch, PAD_IDX)

with torch.no_grad():
    outputs = model(dummy_batch, dummy_mask)

print("  Output shapes:")
for task_name, logits in outputs.items():
    print(f"    {task_name}: {logits.shape}")

# Save encoder information for reproducibility
print("\n" + "-" * 70)
print("ENCODER PARAMETER SUMMARY:")
print("-" * 70)
for arch, encoder in all_encoders.items():
    params = get_encoder_parameter_count(encoder)
    print(f"  {arch:12s}: {params:>10,} parameters  ~{params/1e6:.2f}M")

print("\n✅ Encoders built.")

# Save encoder parameter counts for reference
def save_encoder_info(encoders, filepath="encoder_info.txt"):
    """Save encoder parameter information to a file."""
    with open(filepath, 'w') as f:
        f.write("ENCODER PARAMETER COMPARISON\n")
        f.write("=" * 50 + "\n\n")
        for arch, encoder in encoders.items():
            params = get_encoder_parameter_count(encoder)
            f.write(f"{arch}:\n")
            f.write(f"  Parameters: {params:,}\n")
            f.write(f"  ~{params/1e6:.2f}M\n\n")

        # Add comparison
        base_params = get_encoder_parameter_count(encoders["BiGRU"])
        f.write("COMPARISON:\n")
        for arch, encoder in encoders.items():
            params = get_encoder_parameter_count(encoder)
            diff = abs(params - base_params) / base_params * 100
            f.write(f"  {arch}: {diff:.2f}% difference from BiGRU\n")

# Save information
save_encoder_info(all_encoders)
print(f"\n✅ Encoder information saved to 'encoder_info.txt'")

print("\n" + "=" * 70)
print("READY FOR TRAINING!")
print("=" * 70)
print(f"To use a different encoder, set CONFIG['encoder_type'] to:")
print("  - 'BiGRU'")
print("  - 'Transformer'")


## 4b. Parameter-matched calibration (fair-comparison upgrade)

BiGRU (at its default `hidden_dim`) is the fixed reference. We grid-search Transformer's `dim_feedforward`
so its **encoder body** (all parameters except the shared input
embedding, which is identical across architectures by construction) lands within a small tolerance of
BiGRU's encoder-body parameter count. This gives a `param_matched` condition to run alongside the
`default_hparams` condition, so C1/C3 results aren't confounded by one architecture simply having more
capacity than another.


In [ ]:
# =============================================================================
# 4b. PARAMETER-MATCHED CALIBRATION (~1M PARAMETERS)
# =============================================================================

def count_encoder_body_params(encoder):
    """
    Count only the encoder body parameters (excluding embedding layer).
    This ensures fair comparison between architectures.
    """
    embed_params = sum(p.numel() for p in encoder.embed.parameters())
    total_params = sum(p.numel() for p in encoder.parameters())
    return total_params - embed_params

def get_encoder_total_params(encoder):
    """Get total parameters including embedding layer."""
    return sum(p.numel() for p in encoder.parameters())

def _build_probe(arch, hidden_dim=None, embed_dim=None, n_layers=None, n_heads=None):
    """
    Build a probe encoder for parameter counting.
    embed_dim is always 256 unless explicitly overridden -- kept fixed across
    architectures so the (shared, "identical by construction") input embedding
    table never itself becomes a source of mismatch. Only the encoder BODY
    (dim_feedforward / n_layers for Transformer, hidden_dim for BiGRU) is varied.
    """
    c = CONFIG

    if arch == "BiGRU":
        return BiGRUEncoder(
            VOCAB_SIZE,
            embed_dim or 256,
            hidden_dim or 512,
            n_layers or 3,
            c["dropout"],
            PAD_IDX
        )
    elif arch == "Transformer":
        return TransformerEncoderModel(
            VOCAB_SIZE,
            embed_dim or 256,
            hidden_dim or 512,  # This is dim_feedforward
            n_layers or 3,
            n_heads or 8,
            c["dropout"],
            PAD_IDX
        )
    else:
        raise ValueError(f"Unknown architecture: {arch}")

# =============================================================================
# Build Reference Encoder (BiGRU with ~1M parameters)
# =============================================================================

print("\n" + "=" * 70)
print("PARAMETER-MATCHED CALIBRATION (~1M PARAMETERS)")
print("=" * 70)

# Build reference encoder (BiGRU with ~1M parameters)
reference_encoder = _build_probe("BiGRU", hidden_dim=128, embed_dim=256, n_layers=3)
TARGET_ENCODER_BODY_PARAMS = count_encoder_body_params(reference_encoder)
TARGET_TOTAL_PARAMS = get_encoder_total_params(reference_encoder)

print(f"Reference target (BiGRU):")
print(f"  Total parameters: {TARGET_TOTAL_PARAMS:,} (~{TARGET_TOTAL_PARAMS/1e6:.2f}M)")
print(f"  Encoder body (without embedding): {TARGET_ENCODER_BODY_PARAMS:,}")
print(f"  Configuration: embed_dim=256, hidden_dim=128, n_layers=3")

# =============================================================================
# Calibration Functions
# =============================================================================

def calibrate_grid(arch, target, grid, use_n_layers=False):
    """
    Find the best hyperparameter to match target parameter count.
    embed_dim is always held at 256 (see _build_probe) -- only dim_feedforward
    (default) or n_layers (use_n_layers=True) is varied, so the embedding table
    stays identical to BiGRU's and only the encoder body is being matched.

    Args:
        arch: Architecture name
        target: Target parameter count
        grid: List of values to try
        use_n_layers: If True, grid values are for number of layers

    Returns:
        best_val: Best hyperparameter value
        best_diff: Difference from target
        best_params: Actual parameters at best value
    """
    best_val, best_diff, best_params = None, float("inf"), None

    for val in grid:
        if use_n_layers:
            enc = _build_probe(arch, n_layers=val, embed_dim=256, hidden_dim=512)
        else:
            enc = _build_probe(arch, hidden_dim=val, embed_dim=256, n_layers=3)

        p = count_encoder_body_params(enc)
        diff = abs(p - target) / target if target > 0 else float('inf')

        if diff < best_diff:
            best_diff, best_val, best_params = diff, val, p

    return best_val, best_diff, best_params

# =============================================================================
# Perform Calibration for Transformer (against the BiGRU reference)
# =============================================================================

# Fine-grained dim_feedforward grid for a tighter match than the old 128-step grid
hidden_dim_grid = list(range(32, 800, 8))
n_layers_grid = [1, 2, 3, 4, 5, 6, 7, 8]

print("\n" + "-" * 70)
print("CALIBRATING TRANSFORMER AGAINST THE BiGRU REFERENCE")
print("(embed_dim stays fixed at 256 on both sides -- only dim_feedforward / n_layers vary)")
print("-" * 70)

# Calibrate Transformer via dim_feedforward
print("\n1. Calibrating Transformer via dim_feedforward...")
tf_hidden, tf_diff, tf_params = calibrate_grid(
    "Transformer",
    TARGET_ENCODER_BODY_PARAMS,
    hidden_dim_grid
)
print(f"  Best dim_feedforward: {tf_hidden}")
print(f"  Encoder body params: {tf_params:,}")
print(f"  Difference from target: {tf_diff:.2%}")

# Also try n_layers for Transformer, in case it gives a tighter match
print("\n  Also checking n_layers calibration...")
tf_layers, tf_layers_diff, tf_layers_params = calibrate_grid(
    "Transformer",
    TARGET_ENCODER_BODY_PARAMS,
    n_layers_grid,
    use_n_layers=True
)
print(f"  Best n_layers: {tf_layers}")
print(f"  Encoder body params: {tf_layers_params:,}")
print(f"  Difference from target: {tf_layers_diff:.2%}")

# Choose whichever axis matches tighter -- embed_dim is deliberately NOT a
# candidate axis (see module docstring above): varying it would change the
# input embedding table size too, breaking the "identical shared embedding"
# assumption this whole calibration is built on.
if tf_layers_diff < tf_diff:
    tf_hidden = 512   # dim_feedforward held fixed at 512 while n_layers was searched
    tf_n_layers = tf_layers
    tf_params = tf_layers_params
    tf_diff = tf_layers_diff
    print(f"\n  ✅ Using n_layers={tf_layers}, dim_feedforward=512 (best match: {tf_diff:.2%})")
else:
    tf_n_layers = 3
    print(f"\n  ✅ Using dim_feedforward={tf_hidden}, n_layers=3 (best match: {tf_diff:.2%})")

tf_embed_dim = 256  # always identical to BiGRU's embed_dim

# =============================================================================
# Store Matched Hyperparameters
# =============================================================================

ARCH_MATCHED_HPARAMS = {
    "BiGRU": {
        "hidden_dim": 128,
        "embed_dim": 256,
        "n_layers": 3,
        "params": TARGET_ENCODER_BODY_PARAMS,
        "diff": 0.0
    },
    "Transformer": {
        "hidden_dim": tf_hidden,
        "embed_dim": tf_embed_dim,
        "n_layers": tf_n_layers,
        "params": tf_params,
        "diff": tf_diff,
    },
}

# =============================================================================
# Print Results
# =============================================================================

print("\n" + "-" * 70)
print("PARAM-MATCHED HYPERPARAMETERS")
print("-" * 70)

print(f"  BiGRU:       hidden_dim={CONFIG['hidden_dim']:>4}  params={TARGET_ENCODER_BODY_PARAMS:>10,}  (reference)")
print(f"  Transformer: dim_ff={tf_hidden:>7}  n_layers={tf_n_layers}  params={tf_params:>10,}  (Δ={tf_diff:.2%} from target)")

if tf_diff > 0.15:
    print("\n  ⚠️  NOTE: the match is >15% off target. Consider widening hidden_dim_grid further.")
else:
    print("\n  ✅ Transformer is well-matched to BiGRU (within 15% of target)")

# =============================================================================
# Get Build Kwargs Function
# =============================================================================

def get_build_kwargs(arch, mode):
    """
    Returns build_encoder() override kwargs for a given architecture + comparison mode.

    Args:
        arch: Architecture name ("BiGRU", "Transformer")
        mode: "default_hparams" or "param_matched"

    Returns:
        Dictionary of override parameters
    """
    if mode == "default_hparams":
        return {}
    elif mode == "param_matched":
        hp = ARCH_MATCHED_HPARAMS[arch]
        kwargs = {}

        if hp["hidden_dim"] is not None:
            kwargs["override_hidden_dim"] = hp["hidden_dim"]
        if hp["embed_dim"] is not None:
            kwargs["override_embed_dim"] = hp["embed_dim"]
        if hp["n_layers"] is not None:
            kwargs["override_n_layers"] = hp["n_layers"]

        return kwargs
    else:
        raise ValueError(f"Unknown mode: {mode}. Use 'default_hparams' or 'param_matched'")

# =============================================================================
# Validate Matched Configurations
# =============================================================================

print("\n" + "-" * 70)
print("VALIDATING MATCHED CONFIGURATIONS")
print("-" * 70)

def build_encoder_with_hparams(arch, hparams):
    """Build encoder with given hyperparameters -- uses the same formula the
    calibration search used, so the printed validation matches what Section 7
    actually trains with."""
    if arch == "BiGRU":
        return BiGRUEncoder(
            VOCAB_SIZE,
            hparams["embed_dim"],
            hparams["hidden_dim"],
            hparams["n_layers"],
            CONFIG["dropout"],
            PAD_IDX
        )
    elif arch == "Transformer":
        return TransformerEncoderModel(
            VOCAB_SIZE,
            hparams["embed_dim"],
            hparams["hidden_dim"],
            hparams["n_layers"],
            CONFIG["n_heads"],
            CONFIG["dropout"],
            PAD_IDX
        )

for arch, hparams in ARCH_MATCHED_HPARAMS.items():
    enc = build_encoder_with_hparams(arch, hparams)
    total_params = get_encoder_total_params(enc)
    body_params = count_encoder_body_params(enc)
    diff = abs(body_params - TARGET_ENCODER_BODY_PARAMS) / TARGET_ENCODER_BODY_PARAMS * 100

    print(f"\n{arch}:")
    print(f"  Config: {hparams}")
    print(f"  Total params: {total_params:,} (~{total_params/1e6:.2f}M)")
    print(f"  Body params: {body_params:,}")
    print(f"  Diff from target: {diff:.2f}%")

# =============================================================================
# Build All Encoders with Matched Parameters
# =============================================================================

def build_matched_encoders():
    """Build both encoders with matched parameters."""
    encoders = {}

    print("\n" + "-" * 70)
    print("BUILDING ALL MATCHED ENCODERS")
    print("-" * 70)

    for arch in ["BiGRU", "Transformer"]:
        hparams = ARCH_MATCHED_HPARAMS[arch]
        enc = build_encoder_with_hparams(arch, hparams)
        encoders[arch] = enc
        total_params = get_encoder_total_params(enc)
        print(f"\n{arch}:")
        print(f"  Total parameters: {total_params:,}")

    return encoders

# Build all encoders
matched_encoders = build_matched_encoders()

print("\n" + "=" * 70)
print("✅ PARAMETER CALIBRATION COMPLETE")
print("=" * 70)

print("\nTo use matched parameters:")
print("  kwargs = get_build_kwargs('Transformer', 'param_matched')")
print("  encoder = build_encoder('Transformer', VOCAB_SIZE, PAD_IDX, **kwargs)")

print("\nOr build all encoders:")
print("  encoders = build_matched_encoders()")


## 5. Efficiency measurement utilities (C2)

FLOPs (manual forward-hook estimate), parameter count, latency, peak memory, and GPU energy draw (via `pynvml` when available).


In [ ]:
# =============================================================================
# 5. EFFICIENCY MEASUREMENT (+ ENERGY LOGGING)
# =============================================================================
import time
import numpy as np
import torch
import torch.nn as nn
import os
import pandas as pd

# =============================================================================
# ENCODER FACTORY (used by every experiment below)
# =============================================================================

def build_encoder(name, vocab_size, pad_idx, override_hidden_dim=None,
                 override_embed_dim=None, override_n_layers=None):
    """Build an encoder by name with optional parameter overrides."""
    c = CONFIG

    embed_dim = override_embed_dim if override_embed_dim is not None else c.get("embed_dim", 256)
    n_layers = override_n_layers if override_n_layers is not None else c.get("n_layers", 3)

    if name == "BiGRU":
        hidden_dim = override_hidden_dim if override_hidden_dim is not None else c.get("hidden_dim", 512)
        return BiGRUEncoder(
            vocab_size,
            embed_dim,
            hidden_dim,
            n_layers,
            c.get("dropout", 0.1),
            pad_idx
        )
    elif name == "Transformer":
        hidden_dim = override_hidden_dim if override_hidden_dim is not None else c.get("hidden_dim", 512)
        n_heads = c.get("n_heads", 8)
        return TransformerEncoderModel(
            vocab_size,
            embed_dim,
            hidden_dim,
            n_layers,
            n_heads,
            c.get("dropout", 0.1),
            pad_idx
        )
    else:
        raise ValueError(f"Unknown encoder architecture: {name}")

# =============================================================================
# ENERGY LOGGING SETUP
# =============================================================================

ENERGY_LOGGING_AVAILABLE = False
_GPU_HANDLE = None

if CONFIG.get("enable_energy_logging", False) and CONFIG.get("device", "cpu") == "cuda":
    try:
        import pynvml
        pynvml.nvmlInit()
        _GPU_HANDLE = pynvml.nvmlDeviceGetHandleByIndex(0)
        ENERGY_LOGGING_AVAILABLE = True
        print("✅ GPU energy logging: enabled (pynvml)")
    except Exception as e:
        print(f"⚠️  GPU energy logging: disabled ({e}). Install with `pip install nvidia-ml-py` to enable.")
else:
    print("ℹ️  GPU energy logging: disabled (no CUDA device or disabled in CONFIG)")

def get_gpu_power_watts():
    """Get current GPU power consumption in watts."""
    if not ENERGY_LOGGING_AVAILABLE:
        return float("nan")
    try:
        return pynvml.nvmlDeviceGetPowerUsage(_GPU_HANDLE) / 1000.0
    except Exception:
        return float("nan")

def get_gpu_temperature():
    """Get current GPU temperature in Celsius."""
    if not ENERGY_LOGGING_AVAILABLE:
        return float("nan")
    try:
        return pynvml.nvmlDeviceGetTemperature(_GPU_HANDLE, pynvml.NVML_TEMPERATURE_GPU)
    except Exception:
        return float("nan")

# =============================================================================
# PARAMETER AND FLOPS COUNTING
# =============================================================================

def count_params(model):
    """Count trainable parameters in the model."""
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def estimate_flops_forward(model, sample_input_ids, pad_mask):
    """
    Estimate FLOPs for a forward pass using hooks.
    """
    flops = {"total": 0}
    hooks = []

    # Store batch size and sequence length
    B, L = sample_input_ids.shape

    def linear_hook(module, inp, out):
        # FLOPs for linear layer: 2 * batch * in_features * out_features
        flops["total"] += 2 * inp[0].numel() * module.out_features

    def conv1d_hook(module, inp, out):
        # FLOPs for conv1d: 2 * out_elements * (in_channels / groups) * kernel_size
        out_elems = out.numel()
        in_channels = module.in_channels
        groups = module.groups
        kernel_size = module.kernel_size[0]
        flops["total"] += 2 * out_elems * (in_channels // groups) * kernel_size

    def gru_hook(module, inp, out):
        # GRU hook that handles PackedSequence.
        #
        # For a PackedSequence, inp[0].data.shape[0] is already the TOTAL number of
        # valid (sample, timestep) pairs across the whole batch (i.e. it already
        # embeds both the batch dimension and the per-sample sequence length --
        # padding excluded). Multiplying that by L again double-counts the length
        # dimension and inflates FLOPs by a spurious extra factor of ~L, which is why
        # BiGRU used to come out with ~4x FLOPs per doubling of seq_len instead of the
        # correct ~2x (linear) scaling for a recurrent model.
        nonlocal B, L

        if isinstance(inp[0], torch.nn.utils.rnn.PackedSequence):
            total_valid_steps = inp[0].data.shape[0]
        elif len(inp[0].shape) > 2:
            total_valid_steps = inp[0].shape[0] * inp[0].shape[1]
        else:
            total_valid_steps = inp[0].shape[0] if len(inp[0].shape) > 1 else 1

        input_size = module.input_size
        hidden_size = module.hidden_size
        num_layers = module.num_layers
        num_directions = 2 if module.bidirectional else 1

        # GRU has 3 gates (reset, update, new)
        per_step = 3 * 2 * (input_size + hidden_size) * hidden_size
        flops["total"] += per_step * num_directions * num_layers * total_valid_steps

    # Register hooks for each module type
    for m in model.modules():
        if isinstance(m, nn.Linear):
            hooks.append(m.register_forward_hook(linear_hook))
        elif isinstance(m, nn.Conv1d):
            hooks.append(m.register_forward_hook(conv1d_hook))
        elif isinstance(m, nn.GRU):
            hooks.append(m.register_forward_hook(gru_hook))

    # Run forward pass
    was_training = model.training
    model.eval()
    with torch.no_grad():
        model(sample_input_ids, pad_mask)
    model.train(was_training)

    # Remove hooks
    for h in hooks:
        h.remove()

    return flops["total"]

# =============================================================================
# LATENCY AND ENERGY MEASUREMENT
# =============================================================================

def measure_latency(model, sample_input_ids, pad_mask, n_runs=20, device="cpu", warmup_runs=5):
    """
    Measure inference latency and energy consumption.
    """
    model.eval()
    sample_input_ids = sample_input_ids.to(device)
    pad_mask = pad_mask.to(device)

    # Warmup
    with torch.no_grad():
        for _ in range(warmup_runs):
            model(sample_input_ids, pad_mask)
        if device == "cuda":
            torch.cuda.synchronize()

    # Measure with power readings
    power_readings = []

    with torch.no_grad():
        if device == "cuda":
            torch.cuda.synchronize()

        # Get initial power reading
        power_readings.append(get_gpu_power_watts())
        t0 = time.perf_counter()

        # Run inference
        for _ in range(n_runs):
            model(sample_input_ids, pad_mask)

        if device == "cuda":
            torch.cuda.synchronize()
        t1 = time.perf_counter()

        # Get final power reading
        power_readings.append(get_gpu_power_watts())

    # Calculate metrics
    latency_ms = (t1 - t0) / n_runs * 1000.0
    avg_power_w = np.nanmean(power_readings)

    # Energy in millijoules
    energy_per_call_mj = avg_power_w * (latency_ms / 1000.0) * 1000.0 if not np.isnan(avg_power_w) else float("nan")

    return latency_ms, energy_per_call_mj, avg_power_w

# =============================================================================
# MEMORY MEASUREMENT
# =============================================================================

def measure_peak_memory(model, sample_input_ids, pad_mask, device="cuda"):
    """Measure peak GPU memory usage during inference."""
    if device != "cuda":
        return float("nan")

    model.eval()
    sample_input_ids = sample_input_ids.to(device)
    pad_mask = pad_mask.to(device)

    torch.cuda.reset_peak_memory_stats()
    with torch.no_grad():
        model(sample_input_ids, pad_mask)
        torch.cuda.synchronize()

    return torch.cuda.max_memory_allocated() / (1024 ** 2)

def measure_model_size(model):
    """Measure model size on disk in MB."""
    try:
        import tempfile
        with tempfile.NamedTemporaryFile(suffix='.pt', delete=False) as f:
            torch.save(model.state_dict(), f.name)
            size_mb = os.path.getsize(f.name) / (1024 ** 2)
        os.unlink(f.name)
        return size_mb
    except Exception:
        return float("nan")

# =============================================================================
# COMPREHENSIVE ARCHITECTURE PROFILING
# =============================================================================

def profile_architecture(arch_name, max_len, task_head_sizes, device,
                         build_kwargs=None, n_runs=20, warmup_runs=5):
    """
    Profile a single architecture: params, FLOPs, latency, memory, energy.
    """
    build_kwargs = build_kwargs or {}

    print(f"\nProfiling {arch_name}...")

    # Build encoder and model
    encoder = build_encoder(arch_name, VOCAB_SIZE, PAD_IDX, **build_kwargs)
    model = MultiTaskModel(encoder, task_head_sizes).to(device)

    # Create dummy input
    dummy = torch.randint(0, VOCAB_SIZE, (CONFIG.get("batch_size", 16), max_len))
    dummy_mask = make_pad_mask(dummy, PAD_IDX)

    # Measure parameters
    params = count_params(model)

    # Measure FLOPs
    try:
        flops = estimate_flops_forward(model, dummy.to(device), dummy_mask.to(device))
    except Exception as e:
        print(f"  ⚠️  FLOPs estimation failed: {e}")
        flops = 0

    # Measure latency, energy, power
    latency_ms, energy_mj, power_w = measure_latency(
        model, dummy, dummy_mask,
        n_runs=n_runs,
        warmup_runs=warmup_runs,
        device=device
    )

    # Measure peak memory
    mem_mb = measure_peak_memory(model, dummy, dummy_mask, device=device)

    # Measure model size on disk
    model_size_mb = measure_model_size(model)

    # Get GPU temperature if available
    temp_c = get_gpu_temperature() if device == "cuda" else float("nan")

    results = {
        "arch": arch_name,
        "seq_len": max_len,
        "batch_size": CONFIG.get("batch_size", 16),
        "params": params,
        "params_million": params / 1e6,
        "flops": flops,
        "flops_million": flops / 1e6,
        "latency_ms": latency_ms,
        "energy_per_call_mj": energy_mj,
        "power_watts": power_w,
        "peak_mem_mb": mem_mb,
        "model_size_mb": model_size_mb,
        "temperature_c": temp_c,
        "energy_logging_available": ENERGY_LOGGING_AVAILABLE,
        "device": device
    }

    # Print summary
    print(f"  ✅ {arch_name} profiling complete:")
    print(f"     Parameters: {params:,.0f} ({params/1e6:.2f}M)")
    if flops > 0:
        print(f"     FLOPs: {flops:,.0f} ({flops/1e6:.2f}M)")
    else:
        print(f"     FLOPs: N/A")
    print(f"     Latency: {latency_ms:.2f} ms")
    if not np.isnan(energy_mj):
        print(f"     Energy: {energy_mj:.2f} mJ")
    if not np.isnan(power_w):
        print(f"     Power: {power_w:.2f} W")
    if not np.isnan(mem_mb):
        print(f"     Peak Memory: {mem_mb:.2f} MB")
    if not np.isnan(model_size_mb):
        print(f"     Model Size: {model_size_mb:.2f} MB")

    return results

def profile_all_architectures(seq_lengths=None, device="cuda", n_runs=20, warmup_runs=5):
    """
    Profile both architectures (param-matched) across sequence lengths.
    """
    if seq_lengths is None:
        seq_lengths = [128, 256, 512]

    architectures = ["BiGRU", "Transformer"]
    results = {}

    print("\n" + "=" * 70)
    print("PROFILING ALL ARCHITECTURES")
    print("=" * 70)
    print(f"Device: {device}")
    print(f"Batch size: {CONFIG.get('batch_size', 16)}")
    print(f"Sequence lengths: {seq_lengths}")
    print(f"Runs: {n_runs} (warmup: {warmup_runs})")
    print("-" * 70)

    for arch in architectures:
        results[arch] = {}

        for seq_len in seq_lengths:
            print(f"\n📊 Testing {arch} at seq_len={seq_len}")

            # Get matched parameters for this architecture
            try:
                kwargs = get_build_kwargs(arch, "param_matched")
            except NameError:
                kwargs = {}
                print(f"  ⚠️  get_build_kwargs not found, using default parameters")

            # Profile
            results[arch][seq_len] = profile_architecture(
                arch,
                seq_len,
                task_head_sizes_all,
                device,
                build_kwargs=kwargs,
                n_runs=n_runs,
                warmup_runs=warmup_runs
            )

    return results

# =============================================================================
# RESULTS VISUALIZATION AND SAVING
# =============================================================================

def save_profiling_results(results, filename="profiling_results.csv"):
    """Save profiling results to CSV file."""
    rows = []
    for arch, seq_results in results.items():
        for seq_len, metrics in seq_results.items():
            row = {
                "architecture": arch,
                "sequence_length": seq_len,
                **metrics
            }
            rows.append(row)

    df = pd.DataFrame(rows)
    df.to_csv(filename, index=False)
    print(f"\n✅ Profiling results saved to '{filename}'")
    return df

def print_summary_table(results):
    """Print a formatted summary table of profiling results."""
    print("\n" + "=" * 80)
    print("PROFILING SUMMARY TABLE")
    print("=" * 80)
    print(f"{'Arch':<12} {'Seq Len':<10} {'Params (M)':<12} {'FLOPs (M)':<12} {'Latency (ms)':<14} {'Memory (MB)':<12} {'Energy (mJ)':<12}")
    print("-" * 80)

    for arch, seq_results in results.items():
        for seq_len, metrics in seq_results.items():
            flops_str = f"{metrics['flops_million']:.2f}" if metrics['flops'] > 0 else "N/A"
            energy_str = f"{metrics['energy_per_call_mj']:.2f}" if not np.isnan(metrics['energy_per_call_mj']) else "N/A"
            mem_str = f"{metrics['peak_mem_mb']:.2f}" if not np.isnan(metrics['peak_mem_mb']) else "N/A"
            print(f"{arch:<12} {seq_len:<10} {metrics['params_million']:<12.2f} {flops_str:<12} {metrics['latency_ms']:<14.2f} {mem_str:<12} {energy_str:<12}")

    print("=" * 80)

# =============================================================================
# EXECUTION
# =============================================================================

print("\n" + "=" * 70)
print("EFFICIENCY MEASUREMENT")
print("=" * 70)

# Check device
device = CONFIG.get("device", "cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

# Profile all architectures with different sequence lengths
seq_lengths_to_test = [128, 256, 512]

try:
    profiling_results = profile_all_architectures(
        seq_lengths=seq_lengths_to_test,
        device=device,
        n_runs=20,
        warmup_runs=5
    )

    # Print summary table
    print_summary_table(profiling_results)

    # Save results
    save_profiling_results(profiling_results)

    print("\n" + "=" * 70)
    print("✅ PROFILING COMPLETE")
    print("=" * 70)

except Exception as e:
    print(f"\n❌ Error during profiling: {e}")
    import traceback
    traceback.print_exc()


## 6. Training loops

`train_model` now uses **early stopping on validation loss** (`max_epochs` is an upper bound, not a fixed count) and restores the best-val-loss weights before returning — avoids overfitting to an arbitrary fixed epoch count. Training energy is integrated from per-epoch power sampling.


In [ ]:
# =============================================================================
# 6. TRAINING LOOPS (WITH EARLY STOPPING + ENERGY INTEGRATION)
# =============================================================================
import os
# Reduces CUDA memory fragmentation on small-VRAM GPUs (e.g. 6GB cards) by
# letting the caching allocator grow existing memory segments instead of
# only creating new ones. Must be set before any CUDA context/allocation.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import copy
import time
import numpy as np
import torch
import torch.nn.functional as F
from sklearn.metrics import f1_score, confusion_matrix

def run_epoch(model, loader, task_names, optimizer=None, device="cpu"):
    """
    Run one epoch of training or evaluation.

    Args:
        model: PyTorch model
        loader: DataLoader
        task_names: List of task names
        optimizer: Optimizer (None for evaluation)
        device: Device to run on

    Returns:
        avg_loss: Average loss for the epoch
        acc: Dictionary of accuracy per task
    """
    is_train = optimizer is not None
    model.train(is_train)
    total_loss = 0.0
    correct = {t: 0 for t in task_names}
    total = 0

    for ids, labels_batch in loader:
        ids = ids.to(device)
        pad_mask = make_pad_mask(ids, PAD_IDX)
        labels = {t: labels_batch[t].to(device) for t in task_names}

        if is_train:
            optimizer.zero_grad()

        outputs = model(ids, pad_mask)

        # Compute loss for all tasks
        loss = 0.0
        for t in task_names:
            if t in outputs and t in labels:
                loss = loss + F.cross_entropy(outputs[t], labels[t])

        if is_train:
            loss.backward()
            optimizer.step()

        total_loss += loss.item() * ids.size(0)
        total += ids.size(0)
        for t in task_names:
            if t in outputs:
                correct[t] += (outputs[t].argmax(-1) == labels[t]).sum().item()

    acc = {t: correct[t] / total if total > 0 else 0.0 for t in task_names}
    return total_loss / total if total > 0 else 0.0, acc

def evaluate_full(model, loader, task_names, device):
    """
    Test-time evaluation with accuracy + macro-F1 + confusion matrix per task.

    Args:
        model: PyTorch model
        loader: DataLoader
        task_names: List of task names
        device: Device to run on

    Returns:
        results: Dictionary with accuracy, macro_f1, and confusion matrix per task
    """
    model.eval()
    preds = {t: [] for t in task_names}
    trues = {t: [] for t in task_names}

    with torch.no_grad():
        for ids, labels_batch in loader:
            ids = ids.to(device)
            pad_mask = make_pad_mask(ids, PAD_IDX)
            outputs = model(ids, pad_mask)
            for t in task_names:
                if t in outputs:
                    preds[t].extend(outputs[t].argmax(-1).cpu().tolist())
                    trues[t].extend(labels_batch[t].cpu().tolist())

    results = {}
    for t in task_names:
        if len(trues[t]) == 0:
            results[t] = dict(accuracy=float("nan"), macro_f1=float("nan"), confusion_matrix=None)
            continue

        acc = np.mean(np.array(preds[t]) == np.array(trues[t]))
        macro_f1 = f1_score(trues[t], preds[t], average="macro", zero_division=0)
        cm = confusion_matrix(trues[t], preds[t])
        results[t] = dict(accuracy=acc, macro_f1=macro_f1, confusion_matrix=cm)

    return results

def train_model(model, train_loader, val_loader, task_names, device,
                max_epochs=None, patience=None, lr=None, verbose=True,
                save_best=True, track_energy=True):
    """
    Train the model with early stopping and energy tracking.
    """
    max_epochs = max_epochs or CONFIG.get("max_epochs", 50)
    patience = patience or CONFIG.get("early_stop_patience", 5)
    lr = lr or CONFIG.get("lr", 1e-4)

    model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=CONFIG.get("weight_decay", 1e-5))

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=3
    )

    history = []
    best_val_loss = float("inf")
    best_state = None
    best_epoch = 0
    epochs_no_improve = 0
    total_energy_j = 0.0
    t_start = time.perf_counter()
    epochs_run = 0

    print(f"\nTraining started...")
    print(f"  Device: {device}")
    print(f"  Max epochs: {max_epochs}")
    print(f"  Patience: {patience}")
    print(f"  Learning rate: {lr}")
    print(f"  Tasks: {task_names}")
    print("-" * 70)

    for ep in range(max_epochs):
        # Get power reading before epoch
        power_before = get_gpu_power_watts() if track_energy else float("nan")
        ep_t0 = time.perf_counter()

        # Train and validate
        tr_loss, tr_acc = run_epoch(model, train_loader, task_names, optimizer, device)
        val_loss, val_acc = run_epoch(model, val_loader, task_names, None, device)

        # Update learning rate scheduler
        scheduler.step(val_loss)

        # Calculate energy
        ep_dt = time.perf_counter() - ep_t0
        if track_energy:
            power_after = get_gpu_power_watts() if track_energy else float("nan")
            avg_power = np.nanmean([power_before, power_after])
            if not np.isnan(avg_power):
                total_energy_j += avg_power * ep_dt

        epochs_run += 1

        # Save history
        history_entry = {
            'epoch': ep,
            'train_loss': tr_loss,
            'val_loss': val_loss,
        }
        for t in task_names:
            history_entry[f'train_acc_{t}'] = tr_acc.get(t, 0.0)
            history_entry[f'val_acc_{t}'] = val_acc.get(t, 0.0)
        history.append(history_entry)

        # Print progress
        if verbose:
            acc_str = " ".join(f"val_{t}_acc={val_acc.get(t, 0.0):.3f}" for t in task_names)
            print(f"  epoch {ep+1:3d}/{max_epochs} | train_loss={tr_loss:.4f} val_loss={val_loss:.4f} | {acc_str}")

        # Early stopping check
        if val_loss < best_val_loss - 1e-5:
            best_val_loss = val_loss
            if save_best:
                # Copy to CPU so we don't keep a second full copy of the model pinned on the GPU
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            best_epoch = ep
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                if verbose:
                    print(f"  ⏹️  early stopping at epoch {ep+1} (no val_loss improvement for {patience} epochs)")
                break

    # Restore best model
    if save_best and best_state is not None:
        model.load_state_dict(best_state)
        if verbose:
            print(f"\n  ✅ Restored best model from epoch {best_epoch+1} with val_loss={best_val_loss:.4f}")

    train_time_s = time.perf_counter() - t_start

    print("-" * 70)
    print(f"Training completed:")
    print(f"  Epochs run: {epochs_run}/{max_epochs}")
    print(f"  Best val loss: {best_val_loss:.4f} (epoch {best_epoch+1})")
    print(f"  Total time: {train_time_s:.2f}s")
    if track_energy and total_energy_j > 0:
        print(f"  Total energy: {total_energy_j:.2f}J")
        print(f"  Avg power: {total_energy_j / train_time_s:.2f}W")

    return history, train_time_s, total_energy_j, epochs_run, best_state

def train_and_evaluate_architecture(arch_name, train_loader, val_loader, test_loader,
                                    task_names, device, build_kwargs=None,
                                    max_epochs=None, patience=None, lr=None, verbose=True):
    """
    Train and evaluate a single architecture.
    """
    build_kwargs = build_kwargs or {}

    print("\n" + "=" * 70)
    print(f"TRAINING {arch_name.upper()}")
    print("=" * 70)

    # These are declared here (not inside try) so the `finally` block below can
    # always reach them for cleanup, whether the try block succeeds or crashes
    # partway through (e.g. an OOM during training or during test-set eval).
    encoder = None
    model = None
    result = None

    try:
        # Build model
        encoder = build_encoder(arch_name, VOCAB_SIZE, PAD_IDX, **build_kwargs)
        model = MultiTaskModel(encoder, task_head_sizes_all)

        # Count parameters
        total_params = count_params(model)
        print(f"Total parameters: {total_params:,} ({total_params/1e6:.2f}M)")

        # Train
        history, train_time, energy, epochs_run, best_state = train_model(
            model, train_loader, val_loader, task_names, device,
            max_epochs=max_epochs, patience=patience, lr=lr, verbose=verbose
        )

        # Free training-only memory (optimizer/scheduler already went out of scope
        # inside train_model, but the CUDA caching allocator may still be holding
        # their freed blocks) before the test-set forward pass, which for the
        # Transformer's O(seq_len^2) attention can need a large single allocation.
        if device == "cuda" or (hasattr(device, "type") and device.type == "cuda"):
            torch.cuda.empty_cache()

        # Evaluate on test set
        print("\nEvaluating on test set...")
        test_results = evaluate_full(model, test_loader, task_names, device)

        print("\nTest Results:")
        for task, metrics in test_results.items():
            if not np.isnan(metrics['accuracy']):
                print(f"  {task}:")
                print(f"    Accuracy: {metrics['accuracy']:.4f}")
                print(f"    Macro F1: {metrics['macro_f1']:.4f}")
                if metrics['confusion_matrix'] is not None:
                    print(f"    Confusion Matrix:\n{metrics['confusion_matrix']}")

        result = {
            'architecture': arch_name,
            'history': history,
            'test_results': test_results,
            'train_time': train_time,
            'energy_joules': energy,
            'epochs_run': epochs_run,
            'params': total_params,
            'best_state': best_state,
            'success': True
        }

    except Exception as e:
        print(f"\n❌ Error training {arch_name}: {e}")
        import traceback
        traceback.print_exc()
        result = {
            'architecture': arch_name,
            'success': False,
            'error': str(e)
        }

    finally:
        # --- ALWAYS free GPU memory before the next architecture is built ---
        # This used to live only on the success path. When an exception fired
        # instead (e.g. the Transformer OOM during evaluate_full), execution
        # skipped straight past this cleanup, and the exception's traceback kept
        # `model`/`encoder` and their CUDA tensors referenced from this frame —
        # so the next architecture started with the GPU already full.
        # Putting cleanup in `finally` guarantees it runs on every exit path.
        del model
        del encoder
        import gc
        gc.collect()
        if device == "cuda" or (hasattr(device, "type") and device.type == "cuda"):
            torch.cuda.empty_cache()
            torch.cuda.synchronize()

    return result

def train_all_architectures(train_loader, val_loader, test_loader, task_names, device,
                           max_epochs=None, patience=None, lr=None, verbose=True):
    """
    Train both architectures (param-matched) as a quick side-by-side check.
    """
    architectures = ["BiGRU", "Transformer"]
    all_results = {}

    print("\n" + "=" * 70)
    print("TRAINING ALL ARCHITECTURES")
    print("=" * 70)
    print(f"Device: {device}")
    print(f"Tasks: {task_names}")
    print(f"Max epochs: {max_epochs or CONFIG.get('max_epochs', 50)}")
    print(f"Patience: {patience or CONFIG.get('early_stop_patience', 5)}")

    for arch in architectures:
        try:
            # Get matched parameters
            try:
                kwargs = get_build_kwargs(arch, "param_matched")
            except NameError:
                kwargs = {}
                print(f"  ⚠️  get_build_kwargs not found, using default parameters for {arch}")

            # Train and evaluate
            results = train_and_evaluate_architecture(
                arch, train_loader, val_loader, test_loader, task_names, device,
                build_kwargs=kwargs,
                max_epochs=max_epochs, patience=patience, lr=lr, verbose=verbose
            )
            all_results[arch] = results

        except torch.cuda.OutOfMemoryError as e:
            print(f"\n❌ OOM training {arch}: {e}")
            all_results[arch] = {'architecture': arch, 'success': False, 'error': str(e)}
        except Exception as e:
            print(f"\n❌ Error training {arch}: {e}")
            import traceback
            traceback.print_exc()
            all_results[arch] = {
                'architecture': arch,
                'success': False,
                'error': str(e)
            }

        # Always start the next architecture with a clean slate, whether this one
        # succeeded or crashed with OOM.
        import gc
        gc.collect()
        if device == "cuda" or (hasattr(device, "type") and device.type == "cuda"):
            torch.cuda.empty_cache()

    return all_results

def print_training_summary(all_results):
    """
    Print a summary of training results for all architectures.
    """
    print("\n" + "=" * 80)
    print("TRAINING SUMMARY")
    print("=" * 80)

    print(f"{'Architecture':<15} {'Params (M)':<12} {'Epochs':<10} {'Time (s)':<12} {'Energy (J)':<12} {'Test Acc':<12} {'Status':<10}")
    print("-" * 80)

    for arch, results in all_results.items():
        if not results.get('success', False):
            print(f"{arch:<15} {'ERROR':<12} {'':<10} {'':<12} {'':<12} {'':<12} {'❌':<10}")
            continue

        params_m = results['params'] / 1e6
        epochs = results['epochs_run']
        time_s = results['train_time']
        energy = results['energy_joules'] if results['energy_joules'] > 0 else 0

        # Get average test accuracy
        test_results = results['test_results']
        avg_acc = np.mean([m['accuracy'] for m in test_results.values() if not np.isnan(m['accuracy'])])

        print(f"{arch:<15} {params_m:<12.2f} {epochs:<10} {time_s:<12.2f} {energy:<12.2f} {avg_acc:<12.4f} {'✅':<10}")

    print("=" * 80)

# =============================================================================
# EXECUTION
# =============================================================================

print("\n" + "=" * 70)
print("TRAINING SETUP")
print("=" * 70)

# Check if required variables exist
if 'train_loader' not in locals():
    print("⚠️  train_loader not found. Creating from dataset...")
    # Create dataloaders if not already defined
    try:
        max_len = CONFIG.get("seq_len_file", 512)
        train_loader, val_loader, test_loader = make_loaders(
            train_df, val_df, test_df, max_len, vocab_stoi,
            batch_size=CONFIG.get("batch_size", 64),
            eval_batch_size=CONFIG.get("eval_batch_size", 16),
        )
    except Exception:
        print("❌ Cannot create dataloaders. Make sure dataset is prepared.")

if 'task_names' not in locals():
    task_names = list(TASK_LABEL_COLUMNS.keys())
    print(f"Task names: {task_names}")

# Check device
device = CONFIG.get("device", "cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

# Train all architectures
try:
    all_results = train_all_architectures(
        train_loader, val_loader, test_loader,
        task_names=task_names,
        device=device,
        max_epochs=CONFIG.get("max_epochs", 30),
        patience=CONFIG.get("early_stop_patience", 5),
        lr=CONFIG.get("lr", 1e-4),
        verbose=True
    )

    # Print summary
    print_training_summary(all_results)

    print("\n" + "=" * 70)
    print("✅ TRAINING COMPLETE")
    print("=" * 70)

except Exception as e:
    print(f"\n❌ Error during training: {e}")
    import traceback
    traceback.print_exc()


## 7. Main experiment: C1 — single-task vs multitask, per architecture

For each **(architecture, comparison mode, seed)** we train:
- a single-task model on each of `problem_cls`, `verdict`, `runtime_bucket` individually
- a multitask model on all task heads jointly (shared encoder)

at the file-level sequence length, and record test accuracy + macro-F1 for each. Results are aggregated
across seeds to mean ± std (and a 95% CI), giving the transfer-gain Δ per task per architecture per mode
with an actual uncertainty estimate (C1's central requirement).


In [ ]:
# =============================================================================
# 7. RUN C1 EXPERIMENT: single-task vs multitask, per architecture, per mode, per seed
# =============================================================================
device = CONFIG["device"]
train_loader, val_loader, test_loader = make_loaders(
    train_df, val_df, test_df, CONFIG["seq_len_file"], vocab_stoi,
    batch_size=CONFIG.get("batch_size", 64),
    use_balanced_sampler=True,
    eval_batch_size=CONFIG.get("eval_batch_size", 16),
)

raw_runs = []       # one row per (arch, mode, seed, task-setting) — the audit trail
efficiency_rows = []  # one row per (arch, mode) — efficiency doesn't depend on training seed

for arch in ARCHITECTURES:
    for mode in CONFIG["comparison_modes"]:
        build_kwargs = get_build_kwargs(arch, mode)
        print(f"\n=== Architecture: {arch} | mode: {mode} | build_kwargs: {build_kwargs} ===")

        eff = profile_architecture(arch, CONFIG["seq_len_file"], task_head_sizes_all, device, build_kwargs)
        eff["mode"] = mode
        efficiency_rows.append(eff)
        print(f"  [efficiency] params={eff['params']:,} flops~={eff['flops']:.2e} "
              f"latency={eff['latency_ms']:.2f}ms mem={eff['peak_mem_mb']:.1f}MB "
              f"inference_energy~={eff['energy_per_call_mj']:.2f}mJ")

        for seed in CONFIG["model_seeds"][:CONFIG["n_seeds"]]:
            torch.manual_seed(seed)
            random.seed(seed)
            np.random.seed(seed)
            print(f"  --- seed {seed} ---")

            single_task_acc, single_task_f1 = {}, {}
            for task in TASKS_ALL:
                encoder = build_encoder(arch, VOCAB_SIZE, PAD_IDX, **build_kwargs)
                model = MultiTaskModel(encoder, {task: task_head_sizes_all[task]})
                _, train_time, energy_j, epochs_run, _ = train_model(
                    model, train_loader, val_loader, [task], device, verbose=False)
                test_metrics = evaluate_full(model, test_loader, [task], device)
                single_task_acc[task] = test_metrics[task]["accuracy"]
                single_task_f1[task] = test_metrics[task]["macro_f1"]
                raw_runs.append(dict(arch=arch, mode=mode, seed=seed, setting="single_task", task=task,
                                      accuracy=test_metrics[task]["accuracy"],
                                      macro_f1=test_metrics[task]["macro_f1"],
                                      train_time_s=train_time, energy_j=energy_j, epochs_run=epochs_run))

            encoder = build_encoder(arch, VOCAB_SIZE, PAD_IDX, **build_kwargs)
            model = MultiTaskModel(encoder, task_head_sizes_all)
            _, train_time_mt, energy_j_mt, epochs_run_mt, _ = train_model(
                model, train_loader, val_loader, TASKS_ALL, device, verbose=False)
            test_metrics_mt = evaluate_full(model, test_loader, TASKS_ALL, device)

            acc_str = " ".join(f"{t}={test_metrics_mt[t]['accuracy']:.3f}" for t in TASKS_ALL)
            print(f"     multitask test_acc: {acc_str}  train_time={train_time_mt:.1f}s epochs={epochs_run_mt}")

            for task in TASKS_ALL:
                raw_runs.append(dict(arch=arch, mode=mode, seed=seed, setting="multitask", task=task,
                                      accuracy=test_metrics_mt[task]["accuracy"],
                                      macro_f1=test_metrics_mt[task]["macro_f1"],
                                      train_time_s=train_time_mt, energy_j=energy_j_mt, epochs_run=epochs_run_mt))

raw_runs_df = pd.DataFrame(raw_runs)
eff_df = pd.DataFrame(efficiency_rows)
raw_runs_df.to_csv("c1_raw_runs.csv", index=False)   # full audit trail: every (arch,mode,seed,task) row
eff_df.to_csv("efficiency_profile.csv", index=False)
print("\nSaved raw_runs (audit trail) to c1_raw_runs.csv:", raw_runs_df.shape)
raw_runs_df.head()


## 7b. Aggregate across seeds: mean ± std, 95% CI, transfer-gain Δ


In [ ]:
# =============================================================================
# 7b. AGGREGATE ACROSS SEEDS
# =============================================================================
from scipy import stats as scipy_stats

def mean_ci95(x):
    x = np.asarray(x, dtype=float)
    x = x[~np.isnan(x)]
    n = len(x)
    if n == 0:
        return float("nan"), float("nan")
    m = x.mean()
    if n <= 1:
        return m, float("nan")
    se = x.std(ddof=1) / math.sqrt(n)
    h = se * scipy_stats.t.ppf(0.975, df=n - 1)
    return m, h

agg_rows = []
for (arch, mode, setting, task), g in raw_runs_df.groupby(["arch", "mode", "setting", "task"]):
    acc_mean, acc_ci = mean_ci95(g["accuracy"])
    f1_mean, f1_ci = mean_ci95(g["macro_f1"])
    agg_rows.append(dict(arch=arch, mode=mode, setting=setting, task=task,
                          accuracy_mean=acc_mean, accuracy_ci95=acc_ci,
                          macro_f1_mean=f1_mean, macro_f1_ci95=f1_ci,
                          n_seeds=len(g), accuracy_std=g["accuracy"].std(ddof=0)))
agg_df = pd.DataFrame(agg_rows)

# transfer-gain delta per (arch, mode, task): multitask_mean - single_task_mean, with a paired-seed test
c1_rows = []
for (arch, mode, task), _ in agg_df.groupby(["arch", "mode", "task"]):
    st = raw_runs_df[(raw_runs_df["arch"] == arch) & (raw_runs_df["mode"] == mode) &
                      (raw_runs_df["task"] == task) & (raw_runs_df["setting"] == "single_task")].sort_values("seed")
    mt = raw_runs_df[(raw_runs_df["arch"] == arch) & (raw_runs_df["mode"] == mode) &
                      (raw_runs_df["task"] == task) & (raw_runs_df["setting"] == "multitask")].sort_values("seed")
    st_acc, mt_acc = st["accuracy"].values, mt["accuracy"].values
    delta_per_seed = mt_acc - st_acc
    delta_mean, delta_ci = mean_ci95(delta_per_seed)
    if len(delta_per_seed) >= 2 and np.any(delta_per_seed != delta_per_seed[0]):
        try:
            _, p_val = scipy_stats.wilcoxon(mt_acc, st_acc)
        except Exception:
            p_val = float("nan")
    else:
        p_val = float("nan")
    c1_rows.append(dict(arch=arch, mode=mode, task=task,
                         single_task_acc_mean=st_acc.mean(), multitask_acc_mean=mt_acc.mean(),
                         delta_mean=delta_mean, delta_ci95=delta_ci, delta_wilcoxon_p=p_val,
                         n_seeds=len(delta_per_seed)))

c1_df = pd.DataFrame(c1_rows)
c1_df.to_csv("c1_aggregated.csv", index=False)
print("=== C1 aggregated results (mean over seeds, 95% CI on Δ) ===")
c1_df.round(4)


## 8. Does Δ correlate with architecture family?

Paired tests (Wilcoxon, Friedman) across tasks on the **seed-averaged** Δ, per comparison mode. With 3
tasks this still has limited power — treated as a documented limitation, not overstated as a significant
result unless p is actually low.


In [ ]:
# =============================================================================
# 8. STATISTICAL TEST ON TRANSFER GAIN Δ (per comparison mode)
# =============================================================================
from itertools import combinations

for mode in CONFIG["comparison_modes"]:
    print(f"\n--- comparison mode: {mode} ---")
    sub = c1_df[c1_df["mode"] == mode]
    pivot = sub.pivot(index="task", columns="arch", values="delta_mean")
    print("Δ (multitask - single-task, seed-averaged) per task x architecture:\n", pivot.round(4), "\n")

    print("Pairwise Wilcoxon signed-rank tests on Δ (paired across tasks):")
    for a, b in combinations(ARCHITECTURES, 2):
        try:
            stat, p = scipy_stats.wilcoxon(pivot[a], pivot[b])
            print(f"  {a} vs {b}: W={stat:.3f}, p={p:.4f}")
        except Exception as e:
            print(f"  {a} vs {b}: test not valid with this data ({e})")

    if pivot.shape[0] >= 2:
        try:
            stat, p = scipy_stats.friedmanchisquare(*[pivot[a] for a in ARCHITECTURES])
            print(f"Friedman test across architectures: stat={stat:.3f}, p={p:.4f}")
        except Exception as e:
            print(f"Friedman test skipped ({e})")

print(f"\nCaveat: N={len(TASKS_ALL)} tasks limits statistical power regardless of seed count. "
      "Effect sizes (delta_mean, delta_ci95 in c1_df) are the primary evidence; p-values here are "
      "supporting, not sufficient on their own, for a strong causal claim.")


## 9. Efficiency–Performance Pareto Frontier (C3)

Computed separately per comparison mode, using mean multitask accuracy across seeds.


In [ ]:
# =============================================================================
# 9. PARETO FRONTIER (C3), per comparison mode
# =============================================================================
mt_acc = c1_df[["arch", "mode", "task", "multitask_acc_mean"]].rename(columns={"multitask_acc_mean": "multitask_acc"})
merged = mt_acc.merge(eff_df[["arch", "mode", "flops", "params", "latency_ms", "peak_mem_mb", "energy_per_call_mj"]],
                       on=["arch", "mode"])
merged["gflops"] = merged["flops"] / 1e9
merged["acc_per_gflop"] = merged["multitask_acc"] / merged["gflops"]

def pareto_frontier(df, cost_col, acc_col):
    pts = df[[cost_col, acc_col]].values
    is_dominated = np.zeros(len(pts), dtype=bool)
    for i in range(len(pts)):
        for j in range(len(pts)):
            if i == j:
                continue
            better_or_equal_cost = pts[j][0] <= pts[i][0]
            better_or_equal_acc = pts[j][1] >= pts[i][1]
            strictly_better = pts[j][0] < pts[i][0] or pts[j][1] > pts[i][1]
            if better_or_equal_cost and better_or_equal_acc and strictly_better:
                is_dominated[i] = True
                break
    return ~is_dominated

for mode in CONFIG["comparison_modes"]:
    for task in TASKS_ALL:
        sub = merged[(merged["mode"] == mode) & (merged["task"] == task)].copy()
        if sub.empty:
            continue
        sub["on_frontier"] = pareto_frontier(sub, "gflops", "multitask_acc")
        print(f"\n[{mode}] Task: {task}")
        print(sub[["arch", "gflops", "multitask_acc", "acc_per_gflop", "on_frontier"]]
              .sort_values("gflops").to_string(index=False))

merged.to_csv("pareto_results.csv", index=False)


In [ ]:
# =============================================================================
# 9b. PARETO PLOT (one panel per comparison mode x task)
# =============================================================================
import matplotlib.pyplot as plt

modes = CONFIG["comparison_modes"]
fig, axes = plt.subplots(len(modes), len(TASKS_ALL), figsize=(6 * len(TASKS_ALL), 5 * len(modes)), squeeze=False)
markers = {"BiGRU": "o", "Transformer": "s"}

for mi, mode in enumerate(modes):
    for ti, task in enumerate(TASKS_ALL):
        ax = axes[mi][ti]
        sub = merged[(merged["mode"] == mode) & (merged["task"] == task)]
        for _, row in sub.iterrows():
            ax.scatter(row["gflops"], row["multitask_acc"], s=120, marker=markers.get(row["arch"], "o"),
                       label=row["arch"])
            ax.annotate(row["arch"], (row["gflops"], row["multitask_acc"]),
                        textcoords="offset points", xytext=(6, 6))
        ax.set_xlabel("Compute cost (GFLOPs, forward pass)")
        ax.set_ylabel("Multitask test accuracy (mean over seeds)")
        ax.set_title(f"{mode} — {task}")
        ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("pareto_frontier.png", dpi=150)
plt.show()


## 10. Sequence-length scaling under multitask training (C4)

Sweeps `CONFIG["seq_len_grid"]` (32 → 512 tokens) under `default_hparams`. The `param_matched` mode is reserved
for the main C1 experiment to keep runtime bounded — the calibration in Section 4b is length-independent, so
this is a deliberate scope choice. Each point averages `CONFIG["seq_len_n_seeds"]` seeds (3 by default) and
reports a 95% CI using the same `mean_ci95()` as C1; set it to 1 for a faster single-seed sweep.


In [ ]:
# =============================================================================
# 10. C4: SEQUENCE-LENGTH SCALING EXPERIMENT (DENSE GRID)
# =============================================================================
seq_length_results = []
# Runs CONFIG["seq_len_n_seeds"] seeds per point and aggregates with mean_ci95()
# (Section 7b). Runtime grows linearly with the number of seeds.
seqlen_n_seeds = CONFIG.get("seq_len_n_seeds", 1)
seq_len_seeds = CONFIG["model_seeds"][:seqlen_n_seeds]

for max_len in CONFIG["seq_len_grid"]:
    tr_loader, va_loader, te_loader = make_loaders(
        train_df, val_df, test_df, max_len, vocab_stoi,
        batch_size=CONFIG.get("batch_size", 64),
        use_balanced_sampler=True,
        eval_batch_size=CONFIG.get("eval_batch_size", 16),
    )
    for arch in ARCHITECTURES:
        print(f"\n[seq_len={max_len}] Architecture: {arch}  (seeds: {seq_len_seeds})")
        eff = profile_architecture(arch, max_len, task_head_sizes_all, device)

        per_seed_accs, per_seed_train_times, per_seed_energy = [], [], []
        for seed in seq_len_seeds:
            torch.manual_seed(seed)
            random.seed(seed)
            np.random.seed(seed)
            encoder = build_encoder(arch, VOCAB_SIZE, PAD_IDX)
            model = MultiTaskModel(encoder, task_head_sizes_all)
            _, train_time, energy_j, epochs_run, _ = train_model(
                model, tr_loader, va_loader, TASKS_ALL, device,
                max_epochs=max(4, CONFIG["max_epochs"] // 2), verbose=False)
            test_metrics = evaluate_full(model, te_loader, TASKS_ALL, device)
            seed_mean_acc = np.mean([test_metrics[t]["accuracy"] for t in TASKS_ALL])
            per_seed_accs.append(seed_mean_acc)
            per_seed_train_times.append(train_time)
            per_seed_energy.append(energy_j)

            del model, encoder
            import gc
            gc.collect()
            if device == "cuda" or (hasattr(device, "type") and device.type == "cuda"):
                torch.cuda.empty_cache()

        acc_mean, acc_ci = mean_ci95(per_seed_accs)
        seq_length_results.append(dict(
            seq_len=max_len, arch=arch,
            gflops=eff["flops"] / 1e9, latency_ms=eff["latency_ms"],
            peak_mem_mb=eff["peak_mem_mb"], energy_per_call_mj=eff["energy_per_call_mj"],
            mean_test_acc=acc_mean,
            mean_test_acc_ci95=acc_ci,
            mean_test_acc_std=(float(np.std(per_seed_accs, ddof=0)) if len(per_seed_accs) > 1 else 0.0),
            n_seeds=len(per_seed_accs),
            train_time_s=float(np.mean(per_seed_train_times)),
            energy_j=float(np.mean(per_seed_energy)),
        ))
        ci_str = f"+/-{acc_ci:.3f}" if not np.isnan(acc_ci) else "(n=1, no CI)"
        print(f"  gflops={eff['flops']/1e9:.3f} latency={eff['latency_ms']:.2f}ms mean_acc={acc_mean:.3f} {ci_str}")

seqlen_df = pd.DataFrame(seq_length_results)
seqlen_df.to_csv("sequence_length_scaling.csv", index=False)
print("\n=== Sequence-length scaling summary ===")
seqlen_df


In [ ]:
# =============================================================================
# 10b. CROSSOVER-POINT PLOT + CHECK
# =============================================================================
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for arch in ARCHITECTURES:
    sub = seqlen_df[seqlen_df["arch"] == arch].sort_values("seq_len")
    axes[0].plot(sub["seq_len"], sub["gflops"], marker="o", label=arch)
    axes[1].plot(sub["seq_len"], sub["latency_ms"], marker="o", label=arch)
axes[0].set_xlabel("Sequence length (tokens)"); axes[0].set_ylabel("GFLOPs")
axes[0].set_title("Compute cost vs sequence length"); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].set_xlabel("Sequence length (tokens)"); axes[1].set_ylabel("Latency (ms)")
axes[1].set_title("Latency vs sequence length"); axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig("sequence_length_crossover.png", dpi=150)
plt.show()

print()
prev_cheapest = None
for seq_len in sorted(seqlen_df["seq_len"].unique()):
    sub = seqlen_df[seqlen_df["seq_len"] == seq_len]
    cheapest = sub.loc[sub["gflops"].idxmin(), "arch"]
    fastest = sub.loc[sub["latency_ms"].idxmin(), "arch"]
    flag = "  <-- crossover (cheapest arch changed)" if prev_cheapest and cheapest != prev_cheapest else ""
    print(f"seq_len={seq_len:>4}: lowest-FLOPs={cheapest:<11} lowest-latency={fastest:<11}{flag}")
    prev_cheapest = cheapest


## 11. Practical guidance auto-summary (C5)


In [ ]:
# =============================================================================
# 11. AUTO-GENERATED PRACTICAL GUIDANCE (C5)
# =============================================================================
def generate_guidance(c1_df, merged, seqlen_df):
    lines = ["## Practical guidance (auto-generated from this run's results)\n"]

    for mode in CONFIG["comparison_modes"]:
        sub = c1_df[c1_df["mode"] == mode]
        means = sub.groupby("arch")["delta_mean"].mean().dropna()
        if means.empty:
            lines.append(f"- **[{mode}] Best average positive transfer:** insufficient data (all NaN) — "
                         "check for empty test splits at this scale (see Section 13).")
            continue
        best_arch, best_val = means.idxmax(), means.max()
        lines.append(f"- **[{mode}] Best average positive transfer under multitask training:** "
                     f"architecture `{best_arch}` (mean Delta = {best_val:.4f}).")

    for mode in CONFIG["comparison_modes"]:
        for task in TASKS_ALL:
            sub = merged[(merged["mode"] == mode) & (merged["task"] == task)]
            valid = sub.replace([np.inf, -np.inf], np.nan).dropna(subset=["acc_per_gflop"])
            if valid.empty:
                lines.append(f"- **[{mode}] Best accuracy-per-GFLOP on `{task}`:** insufficient data "
                             "(all NaN/inf) at this scale.")
                continue
            best_frontier = valid.loc[valid["acc_per_gflop"].idxmax(), "arch"]
            lines.append(f"- **[{mode}] Best accuracy-per-GFLOP on `{task}`:** `{best_frontier}`.")

    for seq_len in sorted(seqlen_df["seq_len"].unique()):
        sub = seqlen_df[seqlen_df["seq_len"] == seq_len].dropna(subset=["gflops"])
        if sub.empty:
            continue
        cheapest = sub.loc[sub["gflops"].idxmin(), "arch"]
        lines.append(f"- **Cheapest architecture at seq_len={seq_len}:** `{cheapest}`.")

    lines.append(f"\n- Results aggregated over {CONFIG['n_seeds']} seeds per (architecture, mode, task); "
                 "see `c1_aggregated.csv` for 95% CIs and `c1_raw_runs.csv` for the full audit trail.")
    lines.append("\n- These conclusions are drawn from a pilot of "
                 f"{CONFIG['n_problems']} problems — re-run at larger scale (more problems, more tasks) "
                 "before treating as a final publication result.")
    return "\n".join(lines)

guidance_text = generate_guidance(c1_df, merged, seqlen_df)
print(guidance_text)

with open("practical_guidance.md", "w", encoding="utf-8") as f:
    f.write(guidance_text)


## 12. Reproducibility manifest

Saves library/hardware versions, the exact config used, and seeds, alongside the result CSVs already written — everything needed to audit or reproduce this run.


In [ ]:
# =============================================================================
# 12. REPRODUCIBILITY MANIFEST
# =============================================================================
import sklearn, scipy, matplotlib

manifest = dict(
    run_timestamp=time.strftime("%Y-%m-%d %H:%M:%S"),
    python_version=sys.version,
    platform=platform.platform(),
    torch_version=torch.__version__,
    cuda_available=torch.cuda.is_available(),
    gpu_name=(torch.cuda.get_device_name(0) if torch.cuda.is_available() else None),
    numpy_version=np.__version__,
    pandas_version=pd.__version__,
    sklearn_version=sklearn.__version__,
    scipy_version=scipy.__version__,
    matplotlib_version=matplotlib.__version__,
    config=CONFIG,
    n_problems_selected=len(selected_problem_ids),
    n_submissions_final=len(sample_df),
    vocab_size=VOCAB_SIZE,
    param_matched_hparams=ARCH_MATCHED_HPARAMS,
)

def _json_default(o):
    if isinstance(o, (np.integer,)):
        return int(o)
    if isinstance(o, (np.floating,)):
        return float(o)
    if isinstance(o, (np.ndarray,)):
        return o.tolist()
    return str(o)

with open("run_manifest.json", "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2, default=_json_default)

print("Reproducibility manifest saved to run_manifest.json")
print(json.dumps({k: v for k, v in manifest.items() if k != "config"}, indent=2, default=_json_default))

print("\nAll output files for this run:")
for fname in ["c1_raw_runs.csv", "c1_aggregated.csv", "efficiency_profile.csv", "pareto_results.csv",
              "sequence_length_scaling.csv", "practical_guidance.md", "run_manifest.json",
              "pareto_frontier.png", "sequence_length_crossover.png"]:
    print(f"  {'✓' if os.path.isfile(fname) else '✗ MISSING'}  {fname}")


## 13. Notes, limitations, and how to extend

- **Runtime budget.** The C1 experiment (Section 7) trains `2 architectures × 2 modes × 3 seeds × 4 task-settings`
  = 48 models with early stopping, and Section 10 trains another 30. At 200 problems on an RTX 2060 this is
  roughly a day of GPU time — reduce `comparison_modes` / `n_seeds` for a first pass.
- **Energy numbers are approximate.** Power is sampled before/after each epoch (or each latency benchmark), not
  continuously. For publication-grade energy figures, poll `pynvml` on a background thread at ~1 Hz.
- **The leakage-safe split depends on `user_id`** being populated in the CodeNet metadata. The notebook checks this
  (Section 3) and falls back to a stratified split with a printed warning if it is missing.
- **Statistical power.** 3 tasks is a small N for the paired tests in Section 8, even with multi-seed CIs on each Δ.
  Add task heads via the `TASK_LABEL_COLUMNS` registry in Section 3 — everything downstream adapts automatically.
- **Parameter matching is grid-based**, not a continuous optimiser — check the printed Δ% in Section 4b.
- **Other domains.** Only Sections 1–3 (indexing, tokenizer, dataset) are CodeNet-specific; Sections 4–12 are
  domain-agnostic and can be reused for NLP, time-series or speech.


## 14. Final Summary Dashboard: Baseline vs. All Architectures (Charts)

One consolidated view combining everything computed above: the majority-class
floor, the TF-IDF + Logistic Regression baseline (Section 3b), and every
neural architecture's single-task and multitask accuracy (Section 7b),
faceted by task and comparison mode. A second chart puts baseline and neural
training cost side by side on a log scale, since they differ by orders of
magnitude.

Run this cell last, after Sections 3b and 7b have both executed.


In [ ]:
# =============================================================================
# 14. FINAL DASHBOARD: BASELINE + ALL ARCHITECTURES, ACCURACY + COST
# =============================================================================
import os
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# -------------------------------------------------------------------------
# Try to recover missing DataFrames from CSV files
# -------------------------------------------------------------------------
if "baseline_df" not in globals():
    possible = [
        "baseline_results.csv",
    ]
    for f in possible:
        if os.path.exists(f):
            baseline_df = pd.read_csv(f)
            print(f"Loaded baseline_df from {f}")
            break
if "c1_df" not in globals():
    possible = [
        "c1_aggregated.csv",
    ]
    for f in possible:
        if os.path.exists(f):
            c1_df = pd.read_csv(f)
            print(f"Loaded c1_df from {f}")
            break
if "eff_df" not in globals():
    possible = [
        "efficiency_profile.csv",
    ]
    for f in possible:
        if os.path.exists(f):
            eff_df = pd.read_csv(f)
            print(f"Loaded eff_df from {f}")
            break

# -------------------------------------------------------------------------
# Verify everything exists
# -------------------------------------------------------------------------
missing = []
for name in ["baseline_df", "c1_df", "eff_df"]:
    if name not in globals():
        missing.append(name)
if len(missing):
    print("\nMissing variables:")
    print(missing)
    print("\nCurrent CSV files in this folder:\n")
    for f in sorted(os.listdir(".")):
        if f.endswith(".csv"):
            print("  ", f)
    raise RuntimeError(
        "Dashboard cannot continue because required DataFrames are missing."
    )

# -------------------------------------------------------------------------
# Accuracy Dashboard
# -------------------------------------------------------------------------
modes = CONFIG["comparison_modes"]
bar_colors = {
    "Majority": "#B0B0B0",
    "TFIDF+LogReg": "#8858C8",
    "BiGRU (single)": "#4C72B0",
    "BiGRU (multitask)": "#1F3B6E",
    "Transformer (single)": "#DD8452",
    "Transformer (multitask)": "#9C4A0F",
}
fig, axes = plt.subplots(
    len(modes),
    len(TASKS_ALL),
    figsize=(6 * len(TASKS_ALL), 5 * len(modes)),
    squeeze=False,
)
for mi, mode in enumerate(modes):
    for ti, task in enumerate(TASKS_ALL):
        ax = axes[mi][ti]
        labels = []
        values = []
        colors = []
        # ---------------- Baselines ----------------
        b = baseline_df[baseline_df["task"] == task]
        if not b.empty:
            labels.append("Majority")
            values.append(float(b["majority_baseline"].iloc[0]))
            colors.append(bar_colors["Majority"])
            labels.append("TFIDF+LogReg")
            values.append(float(b["accuracy"].iloc[0]))
            colors.append(bar_colors["TFIDF+LogReg"])
        # ---------------- Neural Models ----------------
        sub = c1_df[
            (c1_df["mode"] == mode) &
            (c1_df["task"] == task)
        ]
        for _, row in sub.iterrows():
            single = f"{row['arch']} (single)"
            multi = f"{row['arch']} (multitask)"
            labels.append(single)
            values.append(float(row["single_task_acc_mean"]))
            colors.append(bar_colors.get(single, "#999999"))
            labels.append(multi)
            values.append(float(row["multitask_acc_mean"]))
            colors.append(bar_colors.get(multi, "#666666"))
        ax.bar(range(len(labels)), values, color=colors)
        ax.set_xticks(range(len(labels)))
        ax.set_xticklabels(labels, rotation=60, ha="right", fontsize=8)
        ax.set_ylim(0, 1)
        ax.set_ylabel("Accuracy")
        ax.set_title(f"{mode} - {task}")
        ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("dashboard_accuracy.png", dpi=200)
plt.show()
print("Saved dashboard_accuracy.png")

# -------------------------------------------------------------------------
# Training Cost
# -------------------------------------------------------------------------
fig2, ax2 = plt.subplots(figsize=(8,5))
labels, values = [], []
if "train_time_s" in baseline_df.columns:
    labels.append("TFIDF+LogReg")
    values.append(baseline_df["train_time_s"].mean())
if "raw_runs_df" not in globals() and os.path.exists("c1_raw_runs.csv"):
    raw_runs_df = pd.read_csv("c1_raw_runs.csv")
if "raw_runs_df" in globals():
    for arch in ARCHITECTURES:
        rows = raw_runs_df[raw_runs_df["arch"] == arch]
        labels.append(arch)
        values.append(rows["train_time_s"].mean())
ax2.bar(labels, values)
ax2.set_yscale("log")
ax2.set_ylabel("Training Time (seconds)")
ax2.set_title("Training Cost Comparison")
ax2.grid(axis="y", which="both", alpha=0.3)
plt.tight_layout()
plt.savefig("dashboard_cost.png", dpi=200)
plt.show()
print("Saved dashboard_cost.png")

# -------------------------------------------------------------------------
# Full comparison table
# -------------------------------------------------------------------------
rows = []
for _, r in baseline_df.iterrows():
    rows.append({
        "model":"Majority",
        "mode":"-",
        "task":r["task"],
        "accuracy":r["majority_baseline"]
    })
    rows.append({
        "model":"TFIDF+LogReg",
        "mode":"baseline",
        "task":r["task"],
        "accuracy":r["accuracy"]
    })
for _, r in c1_df.iterrows():
    rows.append({
        "model":f"{r['arch']} (single)",
        "mode":r["mode"],
        "task":r["task"],
        "accuracy":r["single_task_acc_mean"]
    })
    rows.append({
        "model":f"{r['arch']} (multitask)",
        "mode":r["mode"],
        "task":r["task"],
        "accuracy":r["multitask_acc_mean"]
    })
full_comparison_df = pd.DataFrame(rows)
full_comparison_df = full_comparison_df.sort_values(
    ["task","mode","accuracy"],
    ascending=[True,True,False]
)
full_comparison_df.to_csv(
    "full_comparison_all_models.csv",
    index=False
)
print("\n=== Full Comparison ===\n")
print(full_comparison_df)
